# Attention结构

# DeepSeek V3/V2 Sparse Flash Attention with Quantization

基于 PyPTO 实现的 DeepSeek V3/V2 MLA（Multi-head Latent Attention）稀疏注意力算子，支持 KV Cache INT8 量化和 PagedAttention，运行于华为昇腾 NPU。

## 算法概述

本算子实现了 DeepSeek V3/V2 架构中的稀疏注意力机制，核心特征如下：

- **MLA 压缩表示**：Query/Key 被拆分为 `nope`（低秩压缩部分，`kv_lora_rank=512`）和 `rope`（旋转位置编码部分，`qk_rope_dim=64`）两个子向量，拼接后参与注意力计算。
- **稀疏 Top-K 选择**：每个 query token 仅关注从 KV Cache 中选出的 `topk` 个 key-value 对（默认 topk=2048），而非全序列。
- **KV Cache INT8 量化**：Key 的 `nope` 部分支持 INT8 逐 block 量化（128 元素为一组），配合 FP32 scale 反量化后参与计算。
- **PagedAttention**：KV Cache 以 block（block_size=128）为粒度管理，通过 `block_table` 映射物理位置，支持不连续内存布局。
- **变长序列**：每个 batch 可有不同的实际序列长度（`actual_seq`）。

## 算子规格

| 项目 | 说明 |
|------|------|
| 算子名称 | sparse_flash_attention_quant |
| 数据类型 | BF16 (query/key_nope/key_rope/output), INT8 (可选 key_nope), FP32 (scales) |
| 精度标准 | rtol=0.005, atol=0.0001 |
| 动态轴 | query_nope/query_rope/topk_indices/block_table/kv_act_seqs 的首维为动态 |
| 推理模式 | Decode (s1=1/2, 标准 softmax) / Prefill (s1=256, Flash online softmax) |

### 输入输出

| 参数 | 方向 | shape | dtype | 说明 |
|------|------|-------|-------|------|
| query_nope | 输入 | (B*S1*N_Q, kv_lora_rank) | BF16 | Query 低秩压缩部分 |
| query_rope | 输入 | (B*S1*N_Q, qk_rope_dim) | BF16 | Query 旋转位置编码部分 |
| key_nope_2d | 输入 | (block_num*block_size, kv_lora_rank) | BF16 / INT8 | Key 低秩压缩部分 (KV Cache) |
| key_rope_2d | 输入 | (block_num*block_size, qk_rope_dim) | BF16 | Key 旋转位置编码部分 (KV Cache) |
| k_nope_scales | 输入 | (block_num*block_size, 4) | FP32 | Key INT8 反量化 scale (每 128 元素一组) |
| topk_indices | 输入 | (B*S1, N_KV*topk) | INT32 | 每个 query token 的 top-k 索引 |
| block_table | 输入 | (B, max_blocknum_perbatch) | INT32 | PagedAttention block 映射表 |
| kv_act_seqs | 输入 | (B,) | INT32 | 每个 batch 的实际 KV 序列长度 |
| attention_out | 输出 | (B, S1, N_Q, kv_lora_rank) | BF16 | 注意力计算结果 |

## 实现版本

| 函数 | 模式 | 算法 | 适用芯片 |
|------|------|------|----------|
| `sparse_flash_attention_quant_d` | Decode | 标准 softmax | 910B |
| `sparse_flash_attention_quant_d_950` | Decode | 标准 softmax | 950 |
| `sparse_flash_attention_quant_p` | Prefill | Flash Attention（online softmax） | 910B |

### Decode 模式（`sparse_flash_attention_quant_compute`）

- 使用标准 softmax 归一化：`softmax = exp(S - max(S)) / sum(exp(S - max(S)))`
- 每次 s2 tile 计算后直接得到归一化结果，写入输出
- 适用于 s1 较小（如 s1=1 或 s1=2）的 decode 场景

### Prefill 模式（`sparse_flash_attention_quant_compute_flash`）

- 使用 Flash Attention 算法的 online softmax：维护 `oi_update`（累加输出）、`li_update`（累加归一化因子）、`mi_update`（累加最大值）三个运行状态
- 跨 s2 tile 增量更新：`mi_new = max(mi, tilda_mij)` → 修正历史累加值 → 归一化
- 仅在最后一个 s2 tile 时做最终归一化，减少中间精度损失
- 适用于 s1 较大（如 s1=256）的 prefill 场景

## 实现要点

### 1. 计算流水线（每个 s2 tile）

```
对每个 batch、每个 s1 token、每个 KV head 组:
  ├─ Sa_V0: Gather — 从 KV Cache 按 topk_indices 搬运 Key/Value 数据
  │   ├─ 若 INT8 量化: Gather INT8 kn + scales → 反量化 → BF16
  │   └─ 若 BF16: 直接 Gather BF16 kn
  ├─ Sa_C1: S = Q × K^T (BF16 → FP32 matmul)
  ├─ Sa_V1: Softmax(S * scale)
  │   ├─ Decode: 标准 softmax (exp-max / sum)
  │   └─ Prefill: Flash online softmax (exp-max, 不除 sum, 累积 oi/li/mi)
  ├─ Sa_C2: O = Softmax × V (BF16 matmul)
  └─ Sa_V2: Flash 归一化更新 (仅 Prefill 模式, 最后一个 tile 时 O = oi / li)
```

### 2. 涉及的 PyPTO API

#### 流程控制

| API | 用途 |
|-----|------|
| [`pypto.frontend.jit`](https://gitcode.com/cann/pypto/blob/master/docs/api/config/pypto-frontend-jit.md) | Kernel JIT 编译装饰器，配置 pass_options / runtime_options |
| [`pypto.loop`](https://gitcode.com/cann/pypto/blob/master/docs/api/controlflow/pypto-loop.md) | 生成硬件级循环（batch / s1 / n_kv / group / s2） |
| [`pypto.loop_unroll`](https://gitcode.com/cann/pypto/blob/master/docs/api/controlflow/pypto-loop_unroll.md) | 循环展开（s2 tile 维度） |
| [`pypto.cond`](https://gitcode.com/cann/pypto/blob/master/docs/api/controlflow/pypto-cond.md) | 条件分支（首个/末个 tile 判断） |

#### 张量构造与视图

| API | 用途 |
|-----|------|
| [`pypto.view`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-view.md) | 创建张量视图（切片 topk_indices / block_table / query / key） |
| [`pypto.reshape`](https://gitcode.com/cann/pypto/blob/master/docs/api/tensor/pypto-Tensor-reshape.md) | 张量形状变换（INT8 反量化 reshape 对齐） |
| [`pypto.concat`](https://gitcode.com/cann/pypto/blob/master/docs/api/tensor/pypto-Tensor-concat.md) | 张量拼接（扩展 INT8 列宽用于 reshape） |
| [`pypto.assemble`](https://gitcode.com/cann/pypto/blob/master/docs/api/tensor/pypto-Tensor-assemble.md) | 将子张量写入目标偏移位置（拼接 Key/Query, 写回输出） |

#### 计算算子

| API | 用途 |
|-----|------|
| [`pypto.matmul`](https://gitcode.com/cann/pypto/blob/master/docs/api/tensor/pypto-Tensor-matmul.md) | 矩阵乘法：C1(Q×K^T) 和 C2(Softmax×V) |
| [`pypto.amax`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-amax.md) | 沿指定维度求最大值（Softmax 数值稳定：减最大值防溢出） |
| [`pypto.exp`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-exp.md) | 逐元素指数运算（Softmax 核心） |
| [`pypto.sum`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-sum.md) | 沿指定维度求和（Softmax 归一化因子） |
| [`pypto.maximum`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-maximum.md) | 逐元素取最大值（Flash Attention: mi_new = max(mi, tilda_mij)） |
| [`pypto.mul`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-mul.md) | 逐元素乘法（scale × S, INT8 反量化, Flash 增量修正） |
| [`pypto.sub`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-sub.md) | 逐元素减法（S - max, Flash 增量修正因子） |
| [`pypto.add`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-add.md) | 逐元素加法（Flash 增量更新 li_new, oi_new） |
| [`pypto.div`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-div.md) | 逐元素除法（Softmax 归一化 / Flash 最终归一化 O = oi / li） |
| [`pypto.cast`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-cast.md) | 数据类型转换（INT8→FP16→FP32→BF16 反量化链路） |
| [`pypto.gather`](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-gather.md) | 按 topk_indices 从 KV Cache 搬运数据（gather_in_ub / gather_in_l1 的底层依赖） |

#### Tiling 与编译配置

| API | 用途 |
|-----|------|
| [`pypto.set_vec_tile_shapes`](https://gitcode.com/cann/pypto/blob/master/docs/api/config/pypto-set_vec_tile_shapes.md) | 设置向量算子 tile 尺寸（Gather / Softmax / Flash 更新） |
| [`pypto.set_cube_tile_shapes`](https://gitcode.com/cann/pypto/blob/master/docs/api/config/pypto-set_cube_tile_shapes.md) | 设置 Cube matmul tile 尺寸（C1: Q×K^T, C2: Softmax×V） |
| [`pypto.set_matrix_size`](https://gitcode.com/cann/pypto/blob/master/docs/api/config/pypto-set_matrix_size.md) | 设置 matmul 矩阵尺寸 [M, K, N] |
| [`pypto.set_semantic_label`](https://gitcode.com/cann/pypto/blob/master/docs/api/config/pypto-set_semantic_label.md) | 设置语义标签（Sa_V0 / Sa_C1 / Sa_V1 / Sa_C2 / Sa_V2） |
| [`pypto.set_pass_options`](https://gitcode.com/cann/pypto/blob/master/docs/api/config/pypto-set_pass_options.md) | 编译期 Pass 选项（BF16 路径 scope 隔离） |

### 3. 关键设计决策

#### 5 层嵌套循环结构

Decode 与 Prefill 共享同一循环骨架：
- **L0 batch** → `pypto.loop`, Decode 可并行 (`parallel=True`), Prefill 串行
- **L1 s1** → `pypto.loop`, query 序列维度
- **L2 n_kv** → `pypto.loop`, KV head 维度 (GQA)
- **L3 group** → `pypto.loop`, GQA group 维度 (N_Q / N_KV = 128)
- **L4 s2** → `pypto.loop_unroll`, KV 序列 tile 维度 (unroll_list={1})

#### INT8 量化路径

Key `nope` 部分按每 128 元素分组量化（512 / 128 = 4 组）：
1. `gather_in_ub` 分别搬运 INT8 kn 和 FP32 scales
2. INT8 → FP16 → FP32 类型提升链
3. 逐组乘 scale 完成反量化
4. 转回 BF16 参与后续计算

#### Flash Online Softmax 增量更新

Prefill 模式跨 s2 tile 维护三个运行状态：
- `oi_update`: 累积注意力输出（未归一化）
- `li_update`: 累积 exp sum, shape=(1, group_tile)
- `mi_update`: 累积 max, shape=(1, group_tile)

每个后续 tile 的修正公式：
```
mi_new = max(mi, tilda_mij)
li_new = exp(mi - mi_new) * li + exp(tilda_mij - mi_new) * tilda_lij
oi_new = exp(mi - mi_new) * oi + exp(tilda_mij - mi_new) * q1
```
仅在最后一个 tile 时做最终归一化 `O = oi / li`。

## Tiling 配置

通过 `SaTileShapeConfig` 控制各级计算的 tile 参数：

```python
@dataclass
class SaTileShapeConfig:
    g_tile: int                   # GQA group tile 大小
    s_kv_tile: int                # KV 序列维度 tile 大小
    gather_vec_tile_shape: list   # Gather 向量算子 tile
    c1_tile_shape: list           # C1（Q×K^T）Cube 算子 tile [M0,M1, K0,K1, N0,N1]
    v1_tile_shape: list           # V1（Softmax）向量算子 tile
    c2_tile_shape: list           # C2（Attn×V）Cube 算子 tile [M0,M1, K0,K1, N0,N1]
    v2_tile_shape: list           # V2（Flash 归一化更新）向量算子 tile（仅 prefill 使用）
```

不同芯片/模式的默认配置：

| 参数 | A3 Decode | A3 Prefill | 950 Decode |
|------|-------------|--------------|------------|
| `g_tile` | 128 | 128 | 128 |
| `s_kv_tile` | 2048 | 2048 | 2048 |
| `gather_vec_tile_shape` | [32, 512] | [32, 512] | [64, 512] |
| `c1_tile_shape` | [128,128,128,128,128,128] | [128,128,128,128,128,128] | [128,128,128,128,64,64] |
| `v1_tile_shape` | [8, 2048] | [8, 2048] | [4, 2048] |
| `c2_tile_shape` | [128,128,128,128,128,128] | [128,128,128,128,128,128] | [128,128,128,128,128,128] |
| `v2_tile_shape` | [64, 256] | [64, 128] | [64, 256] |

## 测试用例

| 用例名 | B | S1 | N_Q | N_KV | 序列长度 | Key 量化 | 模式 | 芯片 |
|--------|---|----|-----|------|----------|----------|------|------|
| `sfa_bf16_b4_s2_seq64K_total_int8_d` | 4 | 2 | 128 | 1 | [65536, 16381, 666, 15] | INT8 | Decode | 910B/950 |
| `sfa_bf16_b4_s2_seq64K_per_int8_d` | 4 | 2 | 128 | 1 | [65536]×4 | INT8 | Decode | 910B/950 |
| `sfa_bf16_b4_s2_seq64K_per_bf16_d` | 4 | 2 | 128 | 1 | [65536]×4 | BF16 | Decode | 910B/950 |
| `sfa_bf16_b1_s256_seq64K_int8_p` | 1 | 256 | 128 | 1 | [65536] | INT8 | Prefill | 910B |
| `sfa_bf16_b4_s2_seq64K_per_int8_d_950` | 4 | 2 | 128 | 1 | [65536]×4 | BF16 | Decode | 950 |

精度容差：`atol=0.0001, rtol=0.005`。

## 注意事项

1. **环境要求**：需要可用 NPU 环境（`npu-smi info` 可检测到设备），`TILE_FWK_DEVICE_ID` 环境变量可指定设备编号（默认为 0）。
2. **JIT 编译**：三个 kernel 入口函数通过 `@pypto.frontend.jit` 装饰器注册，首次调用会触发编译。
3. **INT8 量化规则**：Key `nope` 部分按每 128 个元素分组求绝对最大值作为 scale，量化到 `[-128, 127]` 范围。
4. **Gather 算子**：使用 `pypto.experimental.gather_in_ub` / `gather_in_l1` 实现 PagedAttention 的 block 级稀疏索引。
5. **Tiling 调优**：算子性能高度依赖于 `set_vec_tile_shapes` / `set_cube_tile_shapes` 的设置，950 芯片因 UB/L1 容量不同需使用独立的 TileShape 配置。
6. **DYNAMIC Loop**：s2 tile 维度使用 `pypto.loop_unroll(unroll_list={1})`，首次调用时确定循环次数并编译，后续调用可使用更少的迭代次数，但不可超出首次编译时的循环次数。
7. **debug_options**：`sparse_flash_attention_quant_d`（910B Decode）开启了 `runtime_debug_mode` 和 `compile_debug_mode`，正式发布时应移除。

In [1]:
#!/usr/bin/env python3
# coding: utf-8
# Copyright (c) 2025-2026 Huawei Technologies Co., Ltd.
# This program is free software, you can redistribute it and/or modify it under the terms and conditions of
# CANN Open Software License Agreement Version 2.0 (the "License").
# Please refer to the License for details. You may not use this file except in compliance with the License.
# THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
# INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
# See LICENSE in the root of the software repository for the full text of the License.
# -----------------------------------------------------------------------------------------------------------
"""
Sparse Flash Attention Quantization Module — PyPTO Kernel 实现

本模块实现了 DeepSeek V3/V2 MLA 稀疏注意力的 PyPTO kernel，支持：
  - KV Cache INT8 量化（Key nope 部分）与纯 BF16 两种路径
  - PagedAttention（block_table 映射不连续 KV Cache）
  - Top-K 稀疏选择 + Gather 索引
  - 两种 softmax 算法：标准 softmax（Decode）和 Flash online softmax（Prefill）

计算流水线（每个 s2 tile）：
  Sa_V0: Gather — 从 KV Cache 按 topk_indices 搬运 Key/Value 数据，INT8 路径含反量化
  Sa_C1: Cube   — Q × K^T (FP32 matmul)
  Sa_V1: Vector — Softmax（标准归一化 或 Flash 增量更新）
  Sa_C2: Cube   — Softmax × V (BF16 matmul)
  Sa_V2: Vector — Flash 归一化更新（仅 Prefill 模式）

入口函数（@pypto.frontend.jit 装饰，编译后运行于 NPU）：
  - sparse_flash_attention_quant_d     : 910B Decode（标准 softmax）
  - sparse_flash_attention_quant_d_950 : 950  Decode（标准 softmax）
  - sparse_flash_attention_quant_p     : 910B Prefill（Flash online softmax）

依赖：
  - pypto.experimental.gather_in_ub : 从 UB 中按索引 Gather 数据
  - pypto.experimental.gather_in_l1 : 从 L1 中按索引 Gather 数据（Cube 可直接消费）
"""
import os
import math
from dataclasses import dataclass
import numpy as np
import pypto
from pypto.experimental import gather_in_l1, gather_in_ub


@dataclass
class SaTileShapeConfig:
    """
    Sparse Attention 各级算子的 Tile 切分配置。

    默认参数值来自 deepseekv32_sparse_flash_attention_quant.py 中的测试用例配置，
    不同芯片/模式使用不同配置。

    MLA 结构参数（固定）:
        kv_lora_rank = 512 (Key nope / Value 维度, 即 dn)
        qk_rope_dim  = 64  (Key rope 维度, 即 dr)
        d_q = d_k = 576    (dn + dr = 512 + 64)
        topk = 2048        (每个 token 选出的 KV 数量)
        block_size = 128   (PagedAttention 块大小)

    Attributes:
        g_tile: GQA group 维度 tile 大小。
                一次处理的 query head 组数, group = N_Q / N_KV = 128。
                当前配置: 128 (即一次处理全部 128 个 group)
        s_kv_tile: KV 序列维度 tile 大小。
                每个 s2 tile 处理的 KV token 数量。
                当前配置: 2048
        gather_vec_tile_shape: [行, 列] Gather 向量算子 tile。
                控制从 KV Cache Gather 数据时的 UB 切分。
                当前配置: [32, 512] (910B) / [64, 512] (950)
        c1_tile_shape: [M0,M1, K0,K1, N0,N1] C1 (Q×K^T) Cube matmul tile。
                M=Q 侧 group tile, K=K 侧 s2 tile, N=Q 侧 group tile (因 b_trans)。
                当前配置: [128, 128, 128, 128, 128, 128] (910B)
                         [128, 128, 128, 128, 64, 64]   (950, N 轴减半)
        v1_tile_shape: [行, 列] V1 (Softmax) 向量算子 tile。
                控制 softmax 计算（exp/sub/div）的 UB 切分。
                当前配置: [8, 2048] (910B) / [4, 2048] (950)
        c2_tile_shape: [M0,M1, K0,K1, N0,N1] C2 (Softmax×V) Cube matmul tile。
                M=Q 侧 group tile, K=softmax 输出, N=Value 维度。
                注意: C2 的 K 轴必须与 C1 的 N 轴一致。
                当前配置: [128, 128, 128, 128, 128, 128]
        v2_tile_shape: [行, 列] V2 (Flash 归一化更新) 向量算子 tile。
                控制 Flash Attention oi/li/mi 增量更新的 UB 切分。
                仅 Prefill 模式使用。
                当前配置: [64, 128] (Prefill 910B) / [64, 256] (Decode)
    """
    g_tile: int                   # GQA group tile, 默认 128
    s_kv_tile: int                # KV 序列 tile, 默认 2048
    gather_vec_tile_shape: list   # Gather 向量算子 tile, 默认 [32, 512]
    c1_tile_shape: list           # C1 Cube tile [M0,M1,K0,K1,N0,N1], 默认 [128,128,128,128,128,128]
    v1_tile_shape: list           # V1 Softmax tile [行,列], 默认 [8, 2048]
    c2_tile_shape: list           # C2 Cube tile [M0,M1,K0,K1,N0,N1], 默认 [128,128,128,128,128,128]
    v2_tile_shape: list           # V2 Flash 归一化 tile [行,列], 默认 [64, 128]


def sparse_flash_attention_quant_compute(query_nope, query_rope, key_nope_2d, key_rope_2d,
                                         k_nope_scales, topk_indices, block_table, kv_act_seqs,
                                         attention_out, nq, n_kv, softmax_scale, topk,
                                         block_size, max_blocknum_perbatch, tile_config):
    """标准 Softmax 模式 — Decode 阶段专用。

    每个 s2 tile 独立做完整 softmax 归一化后直接写入输出，不使用 Flash 增量更新。
    适用于 s1=1 或 s1=2 的 decode 场景（通常 bn_per_batch=1）。
    被 sparse_flash_attention_quant_d / _d_950 JIT 入口调用。

    计算流水线（每个 s2 tile）：
        Sa_V0: Gather Key (nope+rope) + INT8 反量化（如需）
        Sa_C1: S_ij = Q × K^T           (FP32 Cube matmul)
        Sa_V1: softmax(S * scale)        (标准归一化: exp-max / sum)
        Sa_C2: O = softmax × V           (BF16 Cube matmul)

    Args:
        query_nope:  (t*n_q, kv_lora_rank=512)  BF16, Query 低秩压缩部分
        query_rope:  (t*n_q, qk_rope_dim=64)    BF16, Query 旋转位置编码部分
        key_nope_2d: (total_kv, kv_lora_rank=512) INT8 或 BF16, Key nope 部分 (KV Cache)
        key_rope_2d: (total_kv, qk_rope_dim=64)   BF16, Key rope 部分 (KV Cache)
        k_nope_scales: (total_kv, 4)  FP32, INT8 反量化 scale (每 128 元素一组, 512/128=4)
        topk_indices:  (t, n_kv*topk)  INT32, 每个 query token 的 top-k 索引
        block_table:   (B, max_blocknum_perbatch)  INT32, PagedAttention 块映射表
        kv_act_seqs:   (B,)  INT32, 各 batch 实际 KV 序列长度
        attention_out: (B, S1, N_Q, kv_lora_rank=512)  BF16, 输出
        nq, n_kv: query / kv head 数量
        softmax_scale: 1/sqrt(d_q) = 1/sqrt(576)
        topk: 每个 token 选出的 KV 数量 (2048)
        block_size: PagedAttention 块大小 (128)
        max_blocknum_perbatch: block_table 列数
        tile_config: SaTileShapeConfig tiling 配置
    """
    dtype = query_nope.dtype            # BF16
    kn_dtype = key_nope_2d.dtype        # INT8 或 BF16
    dn = query_nope.shape[1]            # kv_lora_rank = 512
    dr = query_rope.shape[1]            # qk_rope_dim = 64
    group = nq // n_kv                  # GQA group 数, 128/1 = 128
    # 从 tile_config 提取各级 tile 参数
    gather_vec_tile = tile_config.gather_vec_tile_shape   # [32, 512]
    group_tile = tile_config.g_tile                       # 128
    s2_tile = tile_config.s_kv_tile                       # 2048
    c1_tile = tile_config.c1_tile_shape                   # [128,128,128,128,128,128]
    v1_tile = tile_config.v1_tile_shape                   # [8, 2048]
    c2_tile = tile_config.c2_tile_shape                   # [128,128,128,128,128,128]
    n_kv_sym = n_kv                                       # 1

    batch_size_sym = kv_act_seqs.shape[0]                 # B

    s1_n2_gsym = query_nope.shape[0] // batch_size_sym    # S1 * N_Q
    s1_sym = s1_n2_gsym // nq                             # S1

    g_loop_sym = group // group_tile                      # 128/128 = 1

    # 输出中间 tensor: 展平为 2D 方便按 offset 写入
    atten_out_2dim = pypto.tensor([batch_size_sym * s1_n2_gsym, dn], dtype, "attenOut2Dim")

    # ---- LOOP_L0: batch 维度 (可并行) ----
    for batch_idx in pypto.loop(0, batch_size_sym, 1, name="LOOP_L0_idx", idx_name="bIdx", parallel=True):
        cur_act_seq = kv_act_seqs[batch_idx]              # 当前 batch 的实际 KV 序列长度
        # ---- LOOP_L1: s1 (query seq) 维度 ----
        for slc_idx in pypto.loop(0, s1_sym, 1, name="LOOP_L1_s1_SA", idx_name="s1Idx"):
            # 当前 token 可关注的 KV 数量（因果 mask + topk 截断）
            cur_seq = (cur_act_seq - s1_sym + 1 + slc_idx).max(0).min(topk)
            cur_seq.as_variable()
            # s2 tile 数量（向上取整）
            bn_per_batch = (cur_seq + s2_tile - 1) // s2_tile

            # ---- LOOP_L2: n_kv (KV head) 维度 ----
            for n_kv_idx in pypto.loop(0, n_kv_sym, 1, name="LOOP_L2_n_kv_SA", idx_name="n_kvIdx"):
                # ---- LOOP_L3: GQA group 维度 ----
                for group_idx in pypto.loop(0, g_loop_sym, 1, name="LOOP_L3_g_SA", idx_name="gIdx"):
                    cur_group_tile = group_tile            # 当前处理的 group 大小 (128)
                    # 当前 group 在展平 2D query 中的偏移: batch * S1*N_Q + s1*N_Q + n_kv*group + g*tile
                    cur_offset = batch_idx * s1_n2_gsym + slc_idx * nq + n_kv_idx * group + group_idx * cur_group_tile

                    # ---- LOOP_L4: s2 (KV seq tile) 维度 (loop_unroll, unroll_list={1}) ----
                    for s2_idx, _ in pypto.loop_unroll(0, bn_per_batch, 1,
                        name="LOOP_L4_s2_SA", idx_name="s2_idx", unroll_list={1}):
                        cur_s2_tile = s2_tile              # 当前 s2 tile 大小 (2048)

                        # ---- Sa_V0: 准备 Gather 参数 ----
                        # 取出当前 (batch, s1) 的 topk_indices 切片: [1, s2_tile]
                        cur_topk_indices = pypto.view(topk_indices, [1, cur_s2_tile],
                                                [batch_idx * s1_sym + slc_idx, s2_idx * cur_s2_tile],
                                                valid_shape=[1, (cur_seq - s2_idx * cur_s2_tile).min(cur_s2_tile)])
                        # 取出当前 batch 的 block_table 切片: [1, max_blocknum_perbatch]
                        cur_block_table = pypto.view(block_table, [1, max_blocknum_perbatch], [batch_idx, 0])

                        kn = pypto.tensor([s2_tile, dn], dtype, "kn")

                        # ===== INT8 量化 Key 路径 =====
                        if kn_dtype == pypto.DT_INT8:
                            pypto.set_semantic_label("Sa_V0")
                            pypto.set_vec_tile_shapes(16, 1024)  # Vec: 16行 × 1024列, INT8 Gather+Cast 用较大列宽
                            # view scales: (total_kv, 8) 物理列比实际 4 列大，用于对齐
                            k_nope_scale_view = pypto.view(k_nope_scales, [k_nope_scales.shape[0], 8],
                                [0, 0], valid_shape=[k_nope_scales.shape[0], 4])
                            # Gather INT8 scales: 按 topk_indices + block_table 从 KV Cache 搬入 UB
                            kn_scale = gather_in_ub(k_nope_scale_view, cur_topk_indices, cur_block_table,
                                                    block_size, -2)
                            # view INT8 key_nope: (total_kv, 512)
                            k_nope_2d_view = pypto.view(key_nope_2d, [key_nope_2d.shape[0], dn],
                                [0, 0], valid_shape=[key_nope_2d.shape[0], dn])
                            # Gather INT8 quantized kn: 按 topk_indices + block_table 从 KV Cache 搬入 UB
                            kn_quant = gather_in_ub(k_nope_2d_view, cur_topk_indices, cur_block_table, block_size, -2)

                            # 反量化: INT8 → FP16 → FP32 → × scale → BF16
                            kn_quant_fp16 = pypto.cast(kn_quant, pypto.DT_FP16)           # INT8 → FP16
                            kn_quant_fp32 = pypto.cast(kn_quant_fp16, pypto.DT_FP32)      # FP16 → FP32
                            # concat 扩展列: (s2*4, 128) → (s2*4, 256)，用于后续 reshape 对齐
                            kn_quant_fp32 = pypto.concat([kn_quant_fp32, kn_quant_fp32], -1)
                            kn_quant_fp32_tmp = pypto.reshape(kn_quant_fp32, [s2_tile * 8, 128])
                            kn_scale_tmp = pypto.reshape(kn_scale, [s2_tile * 8, 1])
                            # 逐元素乘: INT8_val * scale → FP32 反量化结果
                            pypto.set_vec_tile_shapes(128, 128)  # Vec: 128行 × 128列, INT8 × FP32 scale 逐元素乘
                            kn_fp32 = pypto.mul(kn_quant_fp32_tmp, kn_scale_tmp)
                            kn_fp32_reshape = pypto.reshape(kn_fp32, [s2_tile, dn * 2])
                            # view 取有效区域: (cur_s2_tile, dn)
                            pypto.set_vec_tile_shapes(16, 512)   # Vec: 16行 × 512列, view 反量化结果切有效区域
                            cur_kn_fp32 = pypto.view(kn_fp32_reshape, [cur_s2_tile, dn], [0, 0],
                                valid_shape=[(cur_seq - s2_idx * cur_s2_tile).min(cur_s2_tile), dn])
                            kn = pypto.cast(cur_kn_fp32, dtype)                           # FP32 → BF16

                            # ---- Sa_C1: Q × K^T ----
                            pypto.set_semantic_label("Sa_C1")
                            pypto.set_vec_tile_shapes(gather_vec_tile[0], gather_vec_tile[1])  # Vec: 32行 × 512列 (910B) / 64行 × 512列 (950)
                            pypto.set_cube_tile_shapes([c1_tile[0],
                                c1_tile[1]], [c1_tile[2], c1_tile[3]], [c1_tile[4], c1_tile[5]])
                            # Cube C1: M=[128,128], K=[128,128], N=[128,128] (910B) / N=[64,64] (950)
                            kr = gather_in_l1(key_rope_2d, cur_topk_indices, cur_block_table, block_size, dr,
                                            is_b_matrix=True, is_trans=True)
                            
                            # 拼接 Key: [nope(512) | rope(64)] = 576 维
                            kj = pypto.tensor([cur_s2_tile, dn + dr], dtype, "kj")
                            pypto.assemble(kn, [0, 0], kj)        # kj[:, 0:512] = kn
                            pypto.assemble(kr, [0, dn], kj)       # kj[:, 512:576] = kr
                            kj_view = pypto.view(kj, [cur_s2_tile, dn + dr], [0, 0],
                                valid_shape=[(cur_seq - s2_idx * cur_s2_tile).min(cur_s2_tile), dn + dr])

                            # 拼接 Query: [nope(512) | rope(64)] = 576 维
                            qn = pypto.view(query_nope, [cur_group_tile, dn], [cur_offset, 0],
                                            valid_shape=[cur_group_tile, dn])
                            qr = pypto.view(query_rope, [cur_group_tile, dr], [cur_offset, 0],
                                            valid_shape=[cur_group_tile, dr])
                            qi = pypto.tensor([cur_group_tile, dn + dr], dtype, "qi")
                            pypto.assemble(qn, [0, 0], qi)        # qi[:, 0:512] = q_nope
                            pypto.assemble(qr, [0, dn], qi)       # qi[:, 512:576] = q_rope

                            # C1 matmul: S = Q × K^T, shape=(group_tile, s2_tile), FP32
                            sij = pypto.matmul(qi, kj_view, pypto.DT_FP32, a_trans=False, b_trans=True)

                        # ===== BF16 纯精度 Key 路径 =====
                        else:
                            pypto.set_semantic_label("Sa_V0")
                            pypto.set_vec_tile_shapes(gather_vec_tile[0], gather_vec_tile[1])  # Vec: 32行 × 512列 (910B) / 64行 × 512列 (950)
                            # view BF16 key_nope: (total_kv, 512)
                            k_nope_2d_view = pypto.view(key_nope_2d, [key_nope_2d.shape[0], dn],
                                [0, 0], valid_shape=[key_nope_2d.shape[0], dn])
                            # Gather BF16 kn: 按 topk_indices + block_table 从 KV Cache 搬入 UB
                            kn = gather_in_ub(k_nope_2d_view, cur_topk_indices, cur_block_table, block_size, -2)

                            # ---- Sa_C1: Q × K^T ----
                            pypto.set_semantic_label("Sa_C1")
                            pypto.set_vec_tile_shapes(gather_vec_tile[0], gather_vec_tile[1])  # Vec: 32行 × 512列 (910B) / 64行 × 512列 (950)
                            pypto.set_cube_tile_shapes([c1_tile[0],
                                c1_tile[1]], [c1_tile[2], c1_tile[3]], [c1_tile[4], c1_tile[5]])
                            # Cube C1: M=[128,128], K=[128,128], N=[128,128] (910B) / N=[64,64] (950)
                            key_rope_2d_view = pypto.view(key_rope_2d, [key_rope_2d.shape[0], dr],
                                                            [0, 0], valid_shape=[key_rope_2d.shape[0], dr])
                            kr = gather_in_ub(key_rope_2d_view, cur_topk_indices, cur_block_table, block_size, -2)

                            # 拼接 Key: [nope(512) | rope(64)] = 576 维
                            kj = pypto.tensor([cur_s2_tile, dn + dr], dtype, "kj")
                            pypto.assemble(kn, [0, 0], kj)
                            pypto.assemble(kr, [0, dn], kj)
                            kj_view = pypto.view(kj, [cur_s2_tile, dn + dr], [0, 0],
                                valid_shape=[(cur_seq - s2_idx * cur_s2_tile).min(cur_s2_tile), dn + dr])

                            # 拼接 Query: [nope(512) | rope(64)] = 576 维
                            qn = pypto.view(query_nope, [cur_group_tile, dn], [cur_offset, 0],
                                            valid_shape=[cur_group_tile, dn])
                            qr = pypto.view(query_rope, [cur_group_tile, dr], [cur_offset, 0],
                                            valid_shape=[cur_group_tile, dr])
                            qi = pypto.tensor([cur_group_tile, dn + dr], dtype, "qi")
                            pypto.assemble(qn, [0, 0], qi)
                            pypto.assemble(qr, [0, dn], qi)

                            # C1 matmul: S = Q × K^T, shape=(group_tile, s2_tile), FP32
                            sij = pypto.matmul(qi, kj_view, pypto.DT_FP32, a_trans=False, b_trans=True)

                        # ---- Sa_V1: 标准 Softmax（INT8/BF16 路径汇合） ----
                        pypto.set_semantic_label("Sa_V1")
                        pypto.set_vec_tile_shapes(v1_tile[0], v1_tile[1])  # Vec: 8行 × 2048列 (910B) / 4行 × 2048列 (950)
                        sij_scale = pypto.mul(sij, softmax_scale)                  # S * 1/sqrt(d_q)

                        tilda_mij_reduce = pypto.amax(sij_scale, dim=-1, keepdim=True)
                        t_sub = pypto.sub(sij_scale, tilda_mij_reduce)
                        tilda_pij = pypto.exp(t_sub)
                        tilda_lij_reduce = pypto.sum(tilda_pij, dim=-1, keepdim=True)
                        t_softmax = pypto.div(tilda_pij, tilda_lij_reduce, pypto.PrecisionType.INTRINSIC)
                        tilda_pij_f16 = pypto.cast(t_softmax, dtype)


                        # ---- Sa_C2: softmax × V, shape=(group_tile, dn) ----
                        pypto.set_semantic_label("Sa_C2")
                        pypto.set_cube_tile_shapes([c2_tile[0],
                            c2_tile[1]], [c2_tile[2], c2_tile[3]], [c2_tile[4], c2_tile[5]]) # Cube C2: M=[128,128], K=[128,128], N=[128,128]

                        vj = pypto.view(kn, [cur_s2_tile, dn], [0, 0],
                                        valid_shape=[(cur_seq - s2_idx * cur_s2_tile).min(cur_s2_tile), dn])
                        q1 = pypto.matmul(tilda_pij_f16, vj, dtype)


                        # ---- 写回输出: assemble 到 2D 中间 tensor → reshape 为 4D 输出 ----
                        pypto.assemble(q1, [cur_offset, 0], atten_out_2dim)

                        attention_out[:] = pypto.reshape(atten_out_2dim,
                                                    [attention_out.shape[0], attention_out.shape[1],
                                                     attention_out.shape[2], attention_out.shape[3]], inplace=True)


def sparse_flash_attention_quant_compute_flash(query_nope, query_rope, key_nope_2d, key_rope_2d,
                                               k_nope_scales, topk_indices, block_table, kv_act_seqs,
                                               attention_out, nq, n_kv, softmax_scale, topk,
                                               block_size, max_blocknum_perbatch, tile_config):
    """Flash Online Softmax 模式 — Prefill 阶段专用。

    使用 Flash Attention 算法，跨 s2 tile 增量更新 oi/li/mi 三个运行状态：
      - oi_update: 累积注意力输出 (未归一化)
      - li_update: 累积 exp sum, shape=(1, group_tile)
      - mi_update: 累积 max, shape=(1, group_tile)
    仅在最后一个 s2 tile 时做最终归一化 O_final = oi / li。
    适用于 s1=256 的 prefill 场景。
    被 sparse_flash_attention_quant_p JIT 入口调用。

    计算流水线（每个 s2 tile）：
        Sa_V0: Gather Key (nope+rope) + INT8 反量化（如需）
        Sa_C1: S_ij = Q × K^T           (FP32 Cube matmul)
        Sa_V1: partial softmax           (exp(S-max), 不做最终除法)
        Sa_C2: O_partial = exp_softmax × V (FP32 Cube matmul)
        Sa_UpdateVec2: 增量更新 oi/li/mi (非首个 tile)
        Sa_V2: 最终归一化 O = oi / li    (最后一个 tile)

    Flash Attention 增量更新公式：
        mi_new = max(mi, tilda_mij)
        li_new = exp(mi - mi_new) * li + exp(tilda_mij - mi_new) * tilda_lij
        oi_new = exp(mi - mi_new) * oi + exp(tilda_mij - mi_new) * q1

    Args: 同 sparse_flash_attention_quant_compute，额外使用 tile_config.v2_tile_shape。
    """
    dtype = query_nope.dtype            # BF16
    kn_dtype = key_nope_2d.dtype        # INT8 或 BF16
    dn = query_nope.shape[1]            # kv_lora_rank = 512
    dr = query_rope.shape[1]            # qk_rope_dim = 64
    group = nq // n_kv                  # 128
    # 从 tile_config 提取各级 tile 参数
    group_tile = tile_config.g_tile                       # 128
    s2_tile = tile_config.s_kv_tile                       # 2048
    c1_tile = tile_config.c1_tile_shape                   # [128,128,128,128,128,128]
    v1_tile = tile_config.v1_tile_shape                   # [8, 2048]
    c2_tile = tile_config.c2_tile_shape                   # [128,128,128,128,128,128]
    v2_tile = tile_config.v2_tile_shape                   # [64, 128]
    n_kv_sym = n_kv                                       # 1

    batch_size_sym = kv_act_seqs.shape[0]                 # B

    s1_n2_gsym = query_nope.shape[0] // batch_size_sym    # S1 * N_Q
    s1_sym = s1_n2_gsym // nq                             # S1

    g_loop_sym = group // group_tile                      # 128/128 = 1

    # ---- FLASH_LOOP_L0: batch 维度 ----
    for batch_idx in pypto.loop(0, batch_size_sym, 1, name="FLASH_LOOP_L0_idx", idx_name="bIdx"):
        cur_act_seq = kv_act_seqs[batch_idx]
        # ---- FLASH_LOOP_L1: s1 (query seq) 维度 ----
        for slc_idx in pypto.loop(0, s1_sym, 1, name="FLASH_LOOP_L1_s1_SA", idx_name="s1Idx"):
            # 当前 token 可关注的 KV 数量（因果 mask + topk 截断）
            cur_seq = (cur_act_seq - s1_sym + 1 + slc_idx).max(0).min(topk)
            cur_seq.as_variable()
            # s2 tile 数量（向上取整）
            bn_per_batch = (cur_seq + s2_tile - 1) // s2_tile

            # ---- FLASH_LOOP_L2: n_kv (KV head) 维度 ----
            for n_kv_idx in pypto.loop(0, n_kv_sym, 1, name="FLASH_LOOP_L2_n_kv_SA", idx_name="n_kvIdx"):
                # ---- FLASH_LOOP_L3: GQA group 维度 ----
                for group_idx in pypto.loop(0, g_loop_sym, 1, name="FLASH_LOOP_L3_g_SA", idx_name="gIdx"):
                    cur_group_tile = group_tile            # 128
                    # Flash Attention 三个运行状态: oi (输出), li (exp sum), mi (max)
                    oi_update = pypto.tensor([cur_group_tile, dn], pypto.DT_FP32, "oi_update")  # (128, 512)
                    li_update = pypto.tensor([1, cur_group_tile], pypto.DT_FP32, "li_update")   # (1, 128)
                    mi_update = pypto.tensor([1, cur_group_tile], pypto.DT_FP32, "mi_update")   # (1, 128)

                    # 当前 group 在 2D query 中的偏移
                    cur_offset = batch_idx * s1_n2_gsym + slc_idx * nq + n_kv_idx * group + group_idx * cur_group_tile
                    # 输出 4D attention_out 的写入偏移 [b, s1, head_start, 0]
                    oi_offset = [batch_idx, slc_idx, n_kv_idx * group + group_idx * cur_group_tile, 0]

                    # ---- FLASH_LOOP_L4: s2 (KV seq tile) 维度 (loop_unroll) ----
                    for s2_idx, _ in pypto.loop_unroll(0, bn_per_batch, 1,
                        name="FLASH_LOOP_L4_s2_SA", idx_name="s2_idx", unroll_list={1}):
                        cur_s2_tile = s2_tile              # 2048

                        # ---- Sa_V0: 准备 Gather 参数 ----
                        pypto.set_semantic_label("Sa_V0")
                        # 取出当前 (batch, s1) 的 topk_indices 切片
                        cur_topk_indices = pypto.view(topk_indices, [1, cur_s2_tile],
                                                  [batch_idx * s1_sym + slc_idx, s2_idx * cur_s2_tile],
                                                  valid_shape=[1, (cur_seq - s2_idx * cur_s2_tile).min(cur_s2_tile)])
                        # 取出当前 batch 的 block_table
                        cur_block_table = pypto.view(block_table, [1, max_blocknum_perbatch], [batch_idx, 0])
                        # view key_nope: (total_kv, dn=512)
                        k_nope_2d_view = pypto.view(key_nope_2d, [key_nope_2d.shape[0], dn],
                            [0, 0], valid_shape=[key_nope_2d.shape[0], dn])
                        # view scales: (total_kv, 4)
                        k_nope_scale_view = pypto.view(k_nope_scales, [k_nope_scales.shape[0], 4],
                            [0, 0], valid_shape=[k_nope_scales.shape[0], 4])

                        kn = pypto.tensor([s2_tile, dn], dtype, "kn")

                        # ===== INT8 量化 Key 路径 =====
                        if kn_dtype == pypto.DT_INT8:
                            pypto.set_vec_tile_shapes(32, 512)  # Vec: 32行 × 512列, INT8 Gather scales + quantized kn
                            # Gather INT8 scales
                            kn_scale = gather_in_ub(k_nope_scale_view, cur_topk_indices,
                                                    cur_block_table, block_size, -2)
                            # Gather INT8 quantized kn
                            kn_quant = gather_in_ub(k_nope_2d_view, cur_topk_indices, cur_block_table, block_size, -2)
                            # 反量化: INT8 → FP16 → FP32 → × scale → BF16
                            kn_quant_fp16 = pypto.cast(kn_quant, pypto.DT_FP16)
                            kn_quant_fp32 = pypto.cast(kn_quant_fp16, pypto.DT_FP32)
                            # reshape 为 (s2*4, 128): 512维=4组×128元素
                            kn_quant_fp32_tmp = pypto.reshape(kn_quant_fp32, [s2_tile * 4, 128])
                            kn_scale_tmp = pypto.reshape(kn_scale, [s2_tile * 4, 1])
                            # 逐元素乘: INT8_val * scale
                            pypto.set_vec_tile_shapes(128, 128)  # Vec: 128行 × 128列, INT8 × FP32 scale 逐元素乘
                            kn_fp32 = pypto.mul(kn_quant_fp32_tmp, kn_scale_tmp)
                            kn_fp32_reshape = pypto.reshape(kn_fp32, [s2_tile, dn])
                            # view 有效区域
                            pypto.set_vec_tile_shapes(32, 512)   # Vec: 32行 × 512列, view 反量化结果切有效区域
                            cur_kn_fp32 = pypto.view(kn_fp32_reshape, [cur_s2_tile, dn], [0, 0],
                                valid_shape=[(cur_seq - s2_idx * cur_s2_tile).min(cur_s2_tile), dn])
                            kn = pypto.cast(cur_kn_fp32, dtype)   # FP32 → BF16

                        # ===== BF16 纯精度 Key 路径 =====
                        else:
                            # BF16 kn 直接通过 gather_in_l1 送入 L1，Cube 可直接消费
                            pypto.set_cube_tile_shapes([c1_tile[0], c1_tile[1]],
                                [c1_tile[2], c1_tile[3]], [c1_tile[4], c1_tile[5]])
                            # Cube C1: M=[128,128], K=[128,128], N=[128,128] (910B Prefill)
                            kn = gather_in_l1(key_nope_2d,
                                cur_topk_indices, cur_block_table, block_size, dn, is_b_matrix=True, is_trans=True)

                        # ---- Sa_C1: Q × K^T ----
                        pypto.set_semantic_label("Sa_C1")
                        pypto.set_cube_tile_shapes([c1_tile[0],
                            c1_tile[1]], [c1_tile[2], c1_tile[3]], [c1_tile[4], c1_tile[5]])
                        # Cube C1: M=[128,128], K=[128,128], N=[128,128] (910B Prefill)

                        # Gather key_rope: 送入 L1 供 Cube 消费
                        kr = gather_in_l1(key_rope_2d, cur_topk_indices, cur_block_table, block_size, dr,
                                          is_b_matrix=True, is_trans=True)
                        # 拼接 Key: [nope(512) | rope(64)] = 576 维
                        kj = pypto.tensor([cur_s2_tile, dn + dr], dtype, "kj")
                        pypto.assemble(kn, [0, 0], kj)
                        pypto.assemble(kr, [0, dn], kj)
                        kj_view = pypto.view(kj, [cur_s2_tile, dn + dr], [0, 0],
                                             valid_shape=[(cur_seq - s2_idx * cur_s2_tile).min(cur_s2_tile), dn + dr])

                        # 拼接 Query: [nope(512) | rope(64)] = 576 维
                        qn = pypto.view(query_nope, [cur_group_tile, dn], [cur_offset, 0],
                                        valid_shape=[cur_group_tile, dn])
                        qr = pypto.view(query_rope, [cur_group_tile, dr], [cur_offset, 0],
                                        valid_shape=[cur_group_tile, dr])
                        qi = pypto.tensor([cur_group_tile, dn + dr], dtype, "qi")
                        pypto.assemble(qn, [0, 0], qi)
                        pypto.assemble(qr, [0, dn], qi)

                        # C1 matmul: S = Q × K^T, shape=(group_tile, s2_tile), FP32
                        sij = pypto.matmul(qi, kj_view, pypto.DT_FP32, a_trans=False, b_trans=True)

                        # ---- Sa_V1: Partial Softmax（不做最终除法） ----
                        pypto.set_semantic_label("Sa_V1")
                        pypto.set_vec_tile_shapes(v1_tile[0], v1_tile[1])  # Vec: 8行 × 2048列 (910B Prefill)
                        sij_scale = pypto.mul(sij, softmax_scale)                  # S * 1/sqrt(d_q)
                        tilda_mij_reduce = pypto.amax(sij_scale, dim=-1, keepdim=True)  # (group_tile, 1) 当前 tile max
                        tilda_mij = pypto.reshape(tilda_mij_reduce, [1, cur_group_tile]) # (1, group_tile) 转置用于广播
                        t_sub = pypto.sub(sij_scale, tilda_mij_reduce)              # S - max
                        tilda_pij = pypto.exp(t_sub)                                # exp(S - max), 不除以 sum
                        tilda_pij_f16 = pypto.cast(tilda_pij, dtype)                # → BF16 用于 C2
                        tilda_lij_reduce = pypto.sum(tilda_pij, dim=-1, keepdim=True)   # (group_tile, 1) exp sum
                        tilda_lij = pypto.reshape(tilda_lij_reduce, [1, cur_group_tile]) # (1, group_tile)

                        # ---- Sa_C2: exp_softmax × V ----
                        pypto.set_semantic_label("Sa_C2")
                        pypto.set_cube_tile_shapes([c2_tile[0],
                            c2_tile[1]], [c2_tile[2], c2_tile[3]], [c2_tile[4], c2_tile[5]])
                        # Cube C2: M=[128,128], K=[128,128], N=[128,128] (910B Prefill)
                        pypto.set_matrix_size([tilda_pij_f16.shape[0],
                            tilda_pij_f16.shape[1], kn.shape[1]])

                        # V = Key nope 部分
                        q1 = pypto.tensor([cur_group_tile, dn], dtype)
                        if kn_dtype == pypto.DT_INT8:
                            # INT8 路径: V 从已反量化的 kn 中取
                            vj = pypto.view(kn, [cur_s2_tile, dn], [0, 0],
                                            valid_shape=[(cur_seq - s2_idx * cur_s2_tile).min(cur_s2_tile), dn])
                            q1 = pypto.matmul(tilda_pij_f16, vj, pypto.DT_FP32)
                        else:
                            # BF16 路径: V 通过 gather_in_l1 重新 Gather（不转置）
                            vj = gather_in_l1(key_nope_2d, cur_topk_indices, cur_block_table, block_size,
                                dn, is_b_matrix=True, is_trans=False)
                            q1 = pypto.matmul(tilda_pij_f16, vj, pypto.DT_FP32)

                        # ---- Flash Attention 增量更新 ----
                        if pypto.cond(pypto.is_loop_begin(s2_idx)):
                            # ===== 首个 s2 tile: 初始化 oi/li/mi =====
                            oi_tmp = q1
                            pypto.set_vec_tile_shapes(v2_tile[0], v2_tile[1])  # Vec: 64行 × 128列 (910B Prefill)
                            if pypto.cond(pypto.is_loop_end(s2_idx)):
                                # 只有一个 tile: 直接归一化 O = q1 / li
                                pypto.set_semantic_label("Sa_V2")
                                oi_update[:] = oi_tmp / tilda_lij_reduce
                                # FP32 → BF16, reshape 为 4D 并写入输出
                                pypto.set_vec_tile_shapes(1, 1, v2_tile[0], v2_tile[1])  # Vec: 1×1 + 64×128, cast+assemble 写回输出
                                oi_update_4_dim = pypto.cast(pypto.reshape(oi_update,
                                    [1, 1, cur_group_tile, dn]), dtype)
                                pypto.assemble(oi_update_4_dim, oi_offset, attention_out)
                            else:
                                # 多 tile: 存储未归一化的 q1, 后续 tile 会修正
                                oi_update[:] = oi_tmp
                            pypto.set_vec_tile_shapes(v2_tile[0], v2_tile[1])  # Vec: 64行 × 128列, 更新 li/mi
                            li_update[:] = tilda_lij      # 初始化累积 exp sum
                            mi_update[:] = tilda_mij      # 初始化累积 max
                        else:
                            # ===== 后续 s2 tile: 用 online softmax 公式修正历史值 =====
                            pypto.set_semantic_label("Sa_UpdateVec2")
                            oi = oi_update     # 历史累积 O (未归一化)
                            li = li_update     # 历史累积 exp sum, (1, group_tile)
                            mi = mi_update     # 历史累积 max, (1, group_tile)

                            pypto.set_vec_tile_shapes(v2_tile[0], v2_tile[1])  # Vec: 64行 × 128列, max/sub/exp 修正因子
                            # 新 max = max(历史 max, 当前 tile max)
                            mi_new = pypto.maximum(mi, tilda_mij)
                            # 历史值修正因子: exp(旧max - 新max)
                            t1 = pypto.sub(mi, mi_new)
                            t2 = pypto.exp(t1)
                            # 当前 tile 修正因子: exp(当前max - 新max)
                            t3 = pypto.sub(tilda_mij, mi_new)
                            t4 = pypto.exp(t3)
                            # 更新累积 exp sum: li_new = exp(旧max-新max)*li + exp(当前max-新max)*当前sum
                            t5 = pypto.mul(t4, tilda_lij)
                            t6 = pypto.mul(t2, li)
                            li_new = pypto.add(t6, t5)
                            # 更新 O: oi_new = exp(旧max-新max)*旧oi + exp(当前max-新max)*q1
                            q3 = pypto.mul(oi, pypto.reshape(t2, [cur_group_tile, 1]))    # 修正历史 O
                            pypto.set_vec_tile_shapes(v2_tile[0], v2_tile[1])  # Vec: 64行 × 128列, mul 修正当前 O_partial
                            q2 = pypto.mul(q1, pypto.reshape(t4, [cur_group_tile, 1]))    # 修正当前 O_partial
                            oi_tmp = pypto.add(q3, q2)

                            if pypto.cond(pypto.is_loop_end(s2_idx)):
                                # 最后一个 tile: 最终归一化 O = oi / li
                                oi_update[:] = pypto.div(oi_tmp,
                                    pypto.reshape(li_new, [cur_group_tile, 1]), pypto.PrecisionType.INTRINSIC)
                                # FP32 → BF16, reshape 为 4D 并写入输出
                                pypto.set_vec_tile_shapes(1, 1, v2_tile[0], v2_tile[1])  # Vec: 1×1 + 64×128, cast+assemble 写回输出
                                oi_update_4_dim = pypto.cast(pypto.reshape(oi_update,
                                    [1, 1, cur_group_tile, dn]), dtype)
                                pypto.assemble(oi_update_4_dim, oi_offset, attention_out)
                            else:
                                # 中间 tile: 存储未归一化结果
                                oi_update[:] = oi_tmp
                            li_update[:] = li_new
                            mi_update[:] = mi_new


# ========================================================================================
# JIT 入口函数: @pypto.frontend.jit 装饰，编译后运行于 NPU
# ========================================================================================
#
# pass_options: 编译期 Pass 配置
#   - vec_nbuffer_setting:  向量算子 UB buffer 数量, {-1: 全局默认, 0: 第0个scope, -2: 倒数第2个scope}
#   - cube_l1_reuse_setting: Cube 算子 L1 复用策略, {-1: 全局默认}
#
# runtime_options: 运行时配置
#   - stitch_function_max_num: 最大 stitch 段数 (控制算子分段数上限)
#   - device_sched_mode: 设备调度模式 (3=多核并行调度)
#
# debug_options: 调试配置 (仅开发阶段使用)
#   - runtime_debug_mode: 运行时调试开关
#   - compile_debug_mode: 编译期调试开关
# ========================================================================================


@pypto.frontend.jit(
    pass_options={
        "vec_nbuffer_setting": {-1: 4, -2: 1},      # 全局 4 个 UB buffer, 倒数第 2 scope 1 个
        "cube_l1_reuse_setting": {-1: 8},            # 全局 L1 复用系数 8
    },
    runtime_options={
        "stitch_function_max_num": 128,              # stitch 最大段数 128
        "device_sched_mode": 3                       # 多核并行调度
    }
)
def sparse_flash_attention_quant_d_950(
    query_nope: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),
    query_rope: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),
    key_nope_2d: pypto.Tensor([pypto.STATIC, pypto.STATIC], ), # int8 or bf16
    key_rope_2d: pypto.Tensor([pypto.STATIC, pypto.STATIC], pypto.DT_BF16),
    k_nope_scales: pypto.Tensor([pypto.STATIC, pypto.STATIC], pypto.DT_FP32),
    topk_indices: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_INT32),
    block_table: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_INT32),
    kv_act_seqs: pypto.Tensor([pypto.DYNAMIC], pypto.DT_INT32),
    attention_out: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC, pypto.STATIC, pypto.STATIC], pypto.DT_BF16),

    nq, n_kv, softmax_scale, topk, block_size, max_blocknum_perbatch, tile_config
):
    """950 芯片 Decode 入口 — 标准 softmax。

    调用 sparse_flash_attention_quant_compute（标准 softmax 归一化）。
    适用于 Ascend 950, s1=1/2 的 decode 场景。

    Tensor 类型标注:
        DYNAMIC = 运行时动态维度, STATIC = 编译期固定维度
    """
    pypto.experimental.set_operation_options(combine_axis=True)

    sparse_flash_attention_quant_compute(query_nope, query_rope, key_nope_2d, key_rope_2d,
                                        k_nope_scales, topk_indices, block_table, kv_act_seqs,
                                        attention_out, nq, n_kv, softmax_scale, topk,
                                        block_size, max_blocknum_perbatch, tile_config)


@pypto.frontend.jit(
    pass_options={
        "vec_nbuffer_setting": {-1: 2, 0: 8},       # 全局 2 个 UB buffer, 第 0 scope 8 个
        "cube_l1_reuse_setting": {-1: 2},            # 全局 L1 复用系数 2
    },
    runtime_options={
        "stitch_function_max_num": 128,              # stitch 最大段数 128
        "device_sched_mode": 3                       # 多核并行调度
    },
    debug_options={
        "runtime_debug_mode": 1,                     # 运行时调试开启
        "compile_debug_mode": 1                      # 编译期调试开启
    }
)
def sparse_flash_attention_quant_d(
    query_nope: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),
    query_rope: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),
    key_nope_2d: pypto.Tensor([pypto.STATIC, pypto.STATIC], ), # int8 or bf16
    key_rope_2d: pypto.Tensor([pypto.STATIC, pypto.STATIC], pypto.DT_BF16),
    k_nope_scales: pypto.Tensor([pypto.STATIC, pypto.STATIC], pypto.DT_FP32),
    topk_indices: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_INT32),
    block_table: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_INT32),
    kv_act_seqs: pypto.Tensor([pypto.DYNAMIC], pypto.DT_INT32),
    attention_out: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC, pypto.STATIC, pypto.STATIC], pypto.DT_BF16),

    nq, n_kv, softmax_scale, topk, block_size, max_blocknum_perbatch, tile_config
):
    """910B Decode 入口 — 标准 softmax。

    调用 sparse_flash_attention_quant_compute（标准 softmax 归一化）。
    适用于 Ascend 910B, s1=1/2 的 decode 场景。
    注意: debug_options 仅开发阶段开启，正式发布时应移除。
    """
    pypto.experimental.set_operation_options(combine_axis=True)

    sparse_flash_attention_quant_compute(query_nope, query_rope, key_nope_2d, key_rope_2d,
                                        k_nope_scales, topk_indices, block_table, kv_act_seqs,
                                        attention_out, nq, n_kv, softmax_scale, topk,
                                        block_size, max_blocknum_perbatch, tile_config)


@pypto.frontend.jit(
    pass_options={
        "vec_nbuffer_setting": {-1: 4, 0: 16},      # 全局 4 个 UB buffer, 第 0 scope 16 个 (prefill 需更多 buffer)
        "cube_l1_reuse_setting": {-1: 4},            # 全局 L1 复用系数 4
    },
    runtime_options={
        "stitch_function_max_num": 128               # stitch 最大段数 128 (prefill 不设 device_sched_mode)
    }
)
def sparse_flash_attention_quant_p(
    query_nope: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),
    query_rope: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),
    key_nope_2d: pypto.Tensor([pypto.STATIC, pypto.STATIC],), # int8 or bf16
    key_rope_2d: pypto.Tensor([pypto.STATIC, pypto.STATIC], pypto.DT_BF16),
    k_nope_scales: pypto.Tensor([pypto.STATIC, pypto.STATIC], pypto.DT_FP32),
    topk_indices: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_INT32),
    block_table: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_INT32),
    kv_act_seqs: pypto.Tensor([pypto.DYNAMIC], pypto.DT_INT32),
    attention_out: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC, pypto.STATIC, pypto.STATIC], pypto.DT_BF16),

    nq, n_kv, softmax_scale, topk, block_size, max_blocknum_perbatch, tile_config
):
    """910B Prefill 入口 — Flash online softmax。

    调用 sparse_flash_attention_quant_compute_flash（Flash Attention 增量更新 oi/li/mi）。
    适用于 Ascend 910B, s1=256 的 prefill 场景。
    注意: Prefill 不设 device_sched_mode (与 Decode 不同)。
    """
    pypto.experimental.set_operation_options(combine_axis=True)

    sparse_flash_attention_quant_compute_flash(query_nope, query_rope, key_nope_2d, key_rope_2d,
                                          k_nope_scales, topk_indices, block_table, kv_act_seqs,
                                          attention_out, nq, n_kv, softmax_scale, topk,
                                          block_size, max_blocknum_perbatch, tile_config)


# =============================================
# Test Entry
# =============================================

#!/usr/bin/env python3
# coding: utf-8
# Copyright (c) 2025-2026 Huawei Technologies Co., Ltd.
# This program is free software, you can redistribute it and/or modify it under the terms and conditions of
# CANN Open Software License Agreement Version 2.0 (the "License").
# Please refer to the License for details. You may not use this file except in compliance with the License.
# THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
# INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
# See LICENSE in the root of the software repository for the full text of the License.
# -----------------------------------------------------------------------------------------------------------
"""
DeepSeek V3/V2 Sparse Flash Attention (量化版) 端到端测试

本文件是 DeepSeek MLA 稀疏注意力算子的集成测试入口，职责包括：
  1. 生成测试数据 (golden data)：构造 Query / KV Cache / topk_indices / block_table 等输入，
     支持 INT8 量化和 BF16 两种 Key 格式。
  2. 计算 golden 参考输出：使用纯 PyTorch 实现标准 softmax 和 Flash Attention 两种算法。
  3. 调用 NPU 端 PyPTO kernel 并与 golden 对比验证精度。

支持两种推理模式：
  - Decode (s1=1/2)：标准 softmax，对应 kernel sparse_flash_attention_quant_d / _d_950
  - Prefill (s1=256)：Flash online softmax，对应 kernel sparse_flash_attention_quant_p

支持两种芯片：
  - Ascend 910B
  - Ascend 950（TileShape 不同）

依赖文件：
  - sparse_flash_attention_quant_impl.py : PyPTO kernel 实现（JIT 编译）
  - utils/compare.py                     : 精度对比工具（atol/rtol + NaN/Inf 检测）
"""

# ---- 标准库 ----
import os
import math
import logging
from dataclasses import dataclass

# ---- PyTorch & NPU ----
import torch
import torch_npu

# ---- 测试框架 ----
import pytest
import numpy as np

# ---- PyPTO 算子框架 ----
import pypto

def compare(t, t_ref, name, atol, rtol, max_error_ratio=0.005, max_error_count=10):
    nan_mask = torch.isnan(t)
    nan_count = nan_mask.sum().item()
    inf_mask = torch.isinf(t)
    inf_count = inf_mask.sum().item()
    if nan_count > 0 or inf_count > 0:
        error_msg = f"\n========== 张量 {name} 检测到非法值（禁止存在NaN/Inf）=========="
        if nan_count > 0:
            nan_positions = torch.nonzero(nan_mask, as_tuple=False)
            show_nan_count = min(nan_count, max_error_count)
            error_msg += f"\n- NaN数量：{nan_count}，前 {show_nan_count} 个位置："
            for i in range(show_nan_count):
                pos_tuple = tuple(p.item() for p in nan_positions[i])
                error_msg += f"\n  位置 {pos_tuple}"
        if inf_count > 0:
            inf_positions = torch.nonzero(inf_mask, as_tuple=False)
            show_inf_count = min(inf_count, max_error_count)
            error_msg += f"\n- Inf数量：{inf_count}，前 {show_inf_count} 个位置（值类型）："
            for i in range(show_inf_count):
                pos = inf_positions[i]
                pos_tuple = tuple(p.item() for p in pos)
                inf_val = t[pos_tuple].item()
                inf_type = "+Inf" if inf_val == float('inf') else "-Inf"
                error_msg += f"\n  位置 {pos_tuple}：{inf_type}"
        error_msg += "\n" + "=" * 80 + "\n"
        assert False, error_msg
    assert t.shape == t_ref.shape, f"张量形状不一致：t.shape={t.shape}, t_ref.shape={t_ref.shape}"
    assert t.dtype == t_ref.dtype, f"张量数据类型不一致：t.dtype={t.dtype}, t_ref.dtype={t_ref.dtype}"
    assert t.device == t_ref.device, f"张量设备不一致：t.device={t.device}, t_ref.device={t_ref.device}"
    error_count_threshold = round(max_error_ratio * t_ref.numel())
    diff_abs = (t - t_ref).abs()
    tolerance = atol + rtol * t_ref.abs()
    diff_mask = diff_abs > tolerance
    error_count = diff_mask.sum().item()
    max_diff, flat_max_pos = torch.max(diff_abs.flatten(), dim=0)
    max_pos = torch.unravel_index(flat_max_pos, t.shape)
    max_pos = tuple(idx.item() for idx in max_pos)
    if error_count > 0:
        print(f"\n========== 张量 {name} 存在 {error_count} 个误差点（阈值：{error_count_threshold}）==========")
        error_positions = torch.nonzero(diff_mask, as_tuple=False)
        show_count = min(error_count, max_error_count)
        print(f"显示前 {show_count} 个误差点（位置 | 待比较值 | 参考值 | 绝对误差 | 允许阈值）：")
        for i in range(show_count):
            pos = error_positions[i]
            pos_tuple = tuple(p.item() for p in pos)
            t_val = t[pos_tuple].item()
            t_ref_val = t_ref[pos_tuple].item()
            diff_val = diff_abs[pos_tuple].item()
            tol_val = tolerance[pos_tuple].item()
            print(f"  位置 {pos_tuple}: {t_val:.8f} vs {t_ref_val:.8f} | 误差={diff_val:.8f} | 阈值={tol_val:.8f}")
        print(f"\n最大误差点：位置 {max_pos} | 误差={max_diff.item():.8f} | 阈值={tolerance[max_pos].item():.8f}")
        print("=" * 80 + "\n")
    assert error_count <= error_count_threshold, \
        (f"compare fail: {name}, max diff: {max_diff.item():.8f} at {max_pos}, "
         f"error_count: {error_count}, error_count_threshold: {error_count_threshold}")
    print("compare success !!!!")


def gen_uniform_data(data_shape, min_value, max_value, dtype):
    """
    PyTorch版本的均匀分布数据生成，与NumPy版本行为完全一致
    严格保持 [min_value, max_value) 左闭右开区间特性
    """
    # 特殊情况：全零张量
    if min_value == 0 and max_value == 0:
        return torch.zeros(data_shape, dtype=dtype)
    # 布尔类型处理：等概率生成True/False
    if dtype == torch.bool:
        # 生成[0,2)的整数，转换为bool即等概率True/False
        return torch.randint(0, 2, data_shape, dtype=dtype)
    # 浮点类型：[min_value, max_value)
    if torch.is_floating_point(torch.tensor(0, dtype=dtype)):
        # torch.rand生成[0,1)，缩放后得到[min_value, max_value)
        return min_value + (max_value - min_value) * torch.rand(data_shape, dtype=dtype)
    # 整数类型：[min_value, max_value)
    else:
        # torch.randint的high参数为开区间，直接对应[min_value, max_value)
        return torch.randint(low=min_value, high=max_value, size=data_shape, dtype=dtype)


def compute_attention(input_data, params, s2_tile):
    """
    Flash Attention golden 参考实现（PyTorch）
    使用 online softmax 算法，跨 s2 tile 增量更新 oi/li/mi 三个运行状态，
    仅在最后一个 s2 tile 时做最终归一化，减少中间精度损失。
    对应 PyPTO kernel 中的 sparse_flash_attention_quant_compute_flash（prefill 模式）。

    计算流程（每个 s2 tile）：
        C1: S_ij = Q × K^T           (FP32 matmul)
        V1: online softmax            (exp(S - max), 累积 oi/li/mi)
        C2: O_partial = softmax × V   (FP32 matmul, 在 V1 内完成)
    """
    q, kn, kr, kn_scales, topk_indices, block_table, actual_seq = input_data
    block_size, scalar, topk, d_v, is_kn_quant = params

    # 维度: q=(B,S1,N_Q,D_Q), kn=(total_kv,D_K), kr=(total_kv,D_V)
    # D_Q = kv_lora_rank + qk_rope_dim = 576, D_K = kv_lora_rank = 512, D_V = qk_rope_dim = 64
    b, s1, n1, dq = q.shape
    _, dk = kn.shape
    _, dv = kr.shape

    # topk_indices: (B*S1, topk) — 每个 query token 的 top-k 索引
    if topk_indices.ndim > 2:
        topk_indices = topk_indices.reshape(b * s1, topk)

    atten_out_shape = [b, s1, n1, d_v]
    input_dtype = q.dtype
    kn_dtype = kn.dtype

    # 初始化输出张量
    attention_output = torch.zeros(atten_out_shape, dtype=input_dtype)
    tmp_out = torch.zeros([b, s1, n1], dtype=input_dtype)

    # ---- LOOP_L0: batch 维度 ----
    for b_idx in range(b):
        cur_k_seq = actual_seq[b_idx]
        # ---- LOOP_L1: s1 (query seq) 维度 ----
        for s1_idx in range(s1):
            # 当前 token 实际可关注的 KV 数量（因果 mask + topk 截断）
            cur_seq = min(max(cur_k_seq - s1 + 1 + s1_idx, 0), topk)
            # s2 tile 数量（向上取整）
            bn_per_batch = math.ceil(cur_seq / s2_tile)

            # qi: 当前 (b, s1) 的 query, shape=(N_Q, D_Q)
            qi = q[b_idx, s1_idx, :, :] # (n1, dk)

            # ---- LOOP_L4: s2 (KV seq tile) 维度 ----
            for s2_idx in range(bn_per_batch):
                # 最后一个 tile 可能不足 s2_tile
                s2_tile_cur = min(s2_tile, cur_seq - s2_idx * s2_tile)
                s2_start = s2_tile * s2_idx
                s2_end = s2_start + s2_tile_cur

                # 取出当前 tile 的 topk_indices
                topk_indices_tmp = topk_indices[b_idx * s1 + s1_idx, s2_start:s2_end]

                # 为当前 tile 的 KV 分配临时空间
                slc_kn = torch.zeros([s2_tile_cur, dk], dtype=kn_dtype)
                slc_kr = torch.zeros([s2_tile_cur, dv], dtype=input_dtype)
                slc_kn_scales = torch.zeros([s2_tile_cur, 4], dtype=torch.float32)

                # ---- Gather 阶段: topk_index → KV Cache 物理偏移 ----
                # PagedAttention 映射: topk_index → block_idx → block_table[b, block_idx] → 物理块号
                # 物理偏移 = 物理块号 * block_size + 块内偏移
                offset = torch.zeros([s2_tile_cur], dtype=torch.int32)
                for cur_s2_idx in range(s2_tile_cur):
                    s2_idx_tmp = s2_start + cur_s2_idx
                    topk_index = topk_indices_tmp[s2_idx_tmp]
                    block_idx_in_batch = topk_index // block_size      # 逻辑块索引
                    slc_block_idx = block_table[b_idx, block_idx_in_batch]  # 物理块号
                    tail = topk_index % block_size                     # 块内偏移
                    offset[cur_s2_idx] = slc_block_idx * block_size + tail  # 全局物理偏移

                # ---- 按 offset 从 2D KV Cache 中 Gather 出对应的 kn/kr/scales ----
                for cur_s2_idx in range(s2_tile_cur):
                    slc_idx = offset[cur_s2_idx]
                    slc_kn[cur_s2_idx, :] = kn[slc_idx, :]
                    slc_kr[cur_s2_idx, :] = kr[slc_idx, :]
                    slc_kn_scales[cur_s2_idx, :] = kn_scales[slc_idx, :]

                # ---- 反量化: INT8 Key → BF16 ----
                # kn 按 128 元素一组量化，每组 1 个 scale，共 512/128=4 组
                # 反量化: kn_bf16 = int8_kn * scale (逐组)
                if is_kn_quant:
                    kn_bs = slc_kn.reshape(-1, 128).to(torch.float)       # (s2*4, 128) INT8→FP32
                    kn_scales_tmp = slc_kn_scales.reshape(-1, 1)           # (s2*4, 1) scale
                    kn_tmp = kn_bs * kn_scales_tmp                         # (s2*4, 128) 反量化
                    kn_tmp = kn_tmp.reshape(-1, 512).to(input_dtype)       # (s2, 512) → BF16
                else:
                    kn_tmp = slc_kn
                kr_tmp = slc_kr
                # V = Key 的 nope 部分 (kv_lora_rank=512)
                vj = kn_tmp

                # 拼接 Key: [nope(512) | rope(64)] = 576 维
                kj_view = torch.cat([kn_tmp, kr_tmp], dim=-1)

                # ---- C1: Q × K^T, shape=(N_Q, s2_tile_cur) ----
                sij = torch.matmul(qi.to(torch.float32), kj_view.transpose(1, 0).to(torch.float32)).to(torch.float32)

                # ---- V1: Online Softmax（Flash Attention 核心） ----
                sij_scale = sij * scalar                          # (n1, s2_tile) 乘以 1/sqrt(d_q)
                tilda_mij = sij_scale.amax(dim=-1, keepdims=True) # (n1, 1) 当前 tile 的行最大值
                t_sub = sij_scale - tilda_mij                     # (n1, s2_tile) 减去最大值防溢出
                tilda_pij = torch.exp(t_sub)                      # (n1, s2_tile) exp(S - max)
                tilda_pij_f16 = tilda_pij.to(input_dtype)         # 转 BF16 用于 C2 matmul
                # ---- C2: softmax_exp × V, shape=(N_Q, D_V) ----
                q1 = torch.matmul(tilda_pij_f16.to(torch.float32), vj.to(torch.float32)).to(torch.float32)
                tilda_lij = tilda_pij.sum(dim=-1, keepdims=True)  # (n1, 1) 当前 tile 的 exp 之和

                # ---- Flash Attention 增量更新 ----
                # 首个 s2 tile: 直接初始化 oi/li/mi
                if s2_idx == 0:
                    oi_tmp = q1
                    # 若只有一个 tile，直接归一化
                    if bn_per_batch == 1:
                        oi_update = oi_tmp / tilda_lij
                    else:
                        oi_update = oi_tmp
                    li_update = tilda_lij   # 累积 exp sum
                    mi_update = tilda_mij   # 累积 max
                    tmp_out[b_idx, s1_idx, :] = tilda_lij.reshape(n1)
                    continue

                # 后续 tile: 用 online softmax 公式修正历史累加值
                oi = oi_update     # 历史 O（未归一化）
                li = li_update     # 历史累积 exp sum, shape=(N_Q, 1)
                mi = mi_update     # 历史累积 max, shape=(N_Q, 1)

                # 新 max = max(历史 max, 当前 tile max)
                mi_new = torch.maximum(mi, tilda_mij)
                # 历史值的修正因子: exp(旧max - 新max)
                t1 = mi - mi_new
                t2 = torch.exp(t1)           # 历史指数修正系数
                # 当前 tile 的修正因子: exp(当前max - 新max)
                t3 = tilda_mij - mi_new
                t4 = torch.exp(t3)           # 当前 tile 指数修正系数
                # 更新累积 exp sum: li_new = exp(旧max-新max)*li + exp(当前max-新max)*当前sum
                t5 = t4 * tilda_lij
                t6 = t2 * li
                li_new = t6 + t5
                # 更新 O: oi_new = exp(旧max-新max)*旧oi + exp(当前max-新max)*当前q
                q3 = oi * t2                 # 修正历史 O
                q2 = q1 * t4                 # 修正当前 O_partial
                oi_tmp = q3 + q2
                # 最后一个 tile 时归一化: O_final = oi / li
                if s2_idx == bn_per_batch - 1:
                    oi_update = oi_tmp / li_new
                else:
                    oi_update = oi_tmp
                li_update = li_new
                mi_update = mi_new

            attention_output[b_idx, s1_idx, :, :] = oi_update.to(input_dtype)

    return attention_output, tmp_out


def compute_attention_no_flash(input_data, params, s2_tile):
    """
    标准 Softmax golden 参考实现（PyTorch）
    每个 s2 tile 独立做完整 softmax 归一化后直接得到 O。
    不使用 online softmax 增量更新，适合 s1 较小的 decode 场景。
    对应 PyPTO kernel 中的 sparse_flash_attention_quant_compute（decode 模式）。

    注意: 此实现仅保留最后一个 s2 tile 的计算结果作为输出，
          因为 s2 循环内 atten_out_part 会被覆盖（适合 s1=1/2 的 decode 场景）。

    计算流程（每个 s2 tile）：
        C1: S_ij = Q × K^T           (FP32 matmul)
        V1: 标准 softmax              (exp - max / sum)
        C2: O = softmax × V           (FP32 matmul)
    """
    q, kn, kr, kn_scales, topk_indices, block_table, actual_seq = input_data
    block_size, scalar, topk, d_v, is_kn_quant = params

    # 维度: q=(B,S1,N_Q,D_Q), kn=(total_kv,D_K=512), kr=(total_kv,D_V=64)
    b, s1, n1, dq = q.shape
    _, dk = kn.shape
    _, dv = kr.shape

    # topk_indices: (B*S1, topk)
    if topk_indices.ndim > 2:
        topk_indices = topk_indices.reshape(b * s1, topk)

    atten_out_shape = [b, s1, n1, d_v]
    input_dtype = q.dtype
    kn_dtype = kn.dtype

    # 初始化输出张量
    attention_output = torch.zeros(atten_out_shape, dtype=input_dtype)
    tmp_out = torch.zeros([b, s1, n1], dtype=input_dtype)

    # ---- LOOP_L0: batch 维度 ----
    for b_idx in range(b):
        cur_k_seq = actual_seq[b_idx]
        # ---- LOOP_L1: s1 (query seq) 维度 ----
        for s1_idx in range(s1):
            # 当前 token 实际可关注的 KV 数量（因果 mask + topk 截断）
            cur_seq = min(max(cur_k_seq - s1 + 1 + s1_idx, 0), topk)
            # s2 tile 数量（向上取整）
            bn_per_batch = math.ceil(cur_seq / s2_tile)

            # qi: 当前 (b, s1) 的 query, shape=(N_Q, D_Q)
            qi = q[b_idx, s1_idx, :, :] # (n1, dk)

            # ---- LOOP_L4: s2 (KV seq tile) 维度 ----
            for s2_idx in range(bn_per_batch):
                # 最后一个 tile 可能不足 s2_tile
                s2_tile_cur = min(s2_tile, cur_seq - s2_idx * s2_tile)
                s2_start = s2_tile * s2_idx
                s2_end = s2_start + s2_tile_cur

                # 取出当前 tile 的 topk_indices
                topk_indices_tmp = topk_indices[b_idx * s1 + s1_idx, s2_start:s2_end]

                # 为当前 tile 的 KV 分配临时空间
                slc_kn = torch.zeros([s2_tile_cur, dk], dtype=kn_dtype)
                slc_kr = torch.zeros([s2_tile_cur, dv], dtype=input_dtype)
                slc_kn_scales = torch.zeros([s2_tile_cur, 4], dtype=torch.float32)

                # ---- Gather 阶段: topk_index → KV Cache 物理偏移 ----
                # PagedAttention 映射: topk_index → block_idx → block_table[b, block_idx] → 物理块号
                # 物理偏移 = 物理块号 * block_size + 块内偏移
                offset = torch.zeros([s2_tile_cur], dtype=torch.int32)
                for cur_s2_idx in range(s2_tile_cur):
                    s2_idx_tmp = s2_start + cur_s2_idx
                    topk_index = topk_indices_tmp[s2_idx_tmp]
                    block_idx_in_batch = topk_index // block_size      # 逻辑块索引
                    slc_block_idx = block_table[b_idx, block_idx_in_batch]  # 物理块号
                    tail = topk_index % block_size                     # 块内偏移
                    offset[cur_s2_idx] = slc_block_idx * block_size + tail  # 全局物理偏移

                # ---- 按 offset 从 2D KV Cache 中 Gather 出对应的 kn/kr/scales ----
                for cur_s2_idx in range(s2_tile_cur):
                    slc_idx = offset[cur_s2_idx]
                    slc_kn[cur_s2_idx, :] = kn[slc_idx, :]
                    slc_kr[cur_s2_idx, :] = kr[slc_idx, :]
                    slc_kn_scales[cur_s2_idx, :] = kn_scales[slc_idx, :]

                # ---- 反量化: INT8 Key → BF16 ----
                # kn 按 128 元素一组量化，每组 1 个 scale，共 512/128=4 组
                # 反量化: kn_bf16 = int8_kn * scale (逐组)
                if is_kn_quant:
                    kn_bs = slc_kn.reshape(-1, 128).to(torch.float)       # (s2*4, 128) INT8→FP32
                    kn_scales_tmp = slc_kn_scales.reshape(-1, 1)           # (s2*4, 1) scale
                    kn_tmp = kn_bs * kn_scales_tmp                         # (s2*4, 128) 反量化
                    kn_tmp = kn_tmp.reshape(-1, 512).to(input_dtype)       # (s2, 512) → BF16
                else:
                    kn_tmp = slc_kn
                kr_tmp = slc_kr
                # V = Key 的 nope 部分 (kv_lora_rank=512)
                vj = kn_tmp

                # 拼接 Key: [nope(512) | rope(64)] = 576 维
                kj_view = torch.cat([kn_tmp, kr_tmp], dim=-1)

                # ---- C1: Q × K^T, shape=(N_Q, s2_tile_cur) ----
                sij = torch.matmul(qi.to(torch.float32), kj_view.transpose(1, 0).to(torch.float32)).to(torch.float32)

                # ---- V1: 标准 Softmax ----
                sij_scale = sij * scalar                          # (n1, s2_tile) 乘以 1/sqrt(d_q)
                tilda_mij = sij_scale.amax(dim=-1, keepdims=True) # (n1, 1) 行最大值（防溢出）
                t_sub = sij_scale - tilda_mij                     # (n1, s2_tile) 减最大值
                tilda_pij = torch.exp(t_sub)                      # (n1, s2_tile) exp(S - max)
                tilda_lij = tilda_pij.sum(dim=-1, keepdims=True)  # (n1, 1) sum of exp
                # softmax = exp / sum, 转回 BF16
                tmp_softmax = (tilda_pij / tilda_lij).to(input_dtype)
                # ---- C2: softmax × V, shape=(N_Q, D_V) ----
                atten_out_part = torch.matmul(tmp_softmax.to(torch.float32), vj.to(torch.float32)).to(torch.float32)

            # 注意: decode 场景 s1=1 或 s1=2, bn_per_batch 通常为 1,
            #       所以最后一个 s2 tile 的结果即为最终输出
            attention_output[b_idx, s1_idx, :, :] = atten_out_part.to(input_dtype)

    return attention_output, tmp_out


def gen_block_table(act_seq, block_size, s1, need_indices=False):
    """
    生成 PagedAttention 的 block_table，模拟 KV Cache 的不连续内存布局。
    block_table[b, j] = 物理块号，表示 batch b 的第 j 个逻辑块映射到哪个物理块。
    物理块号随机排列以模拟真实场景。

    Args:
        act_seq: 各 batch 的实际 KV 序列长度, shape=(B,)
        block_size: 每个逻辑块的大小（默认 128）
        s1: query 序列长度（need_indices=True 时用于计算 cache_index）
        need_indices: 是否同时返回全局 cache_index（用于直接索引 KV Cache）

    Returns:
        block_num: 总物理块数（所有 batch 之和）
        block_table: shape=(B, max_blocknum_perbatch), dtype=INT32
        cache_index: shape=(B, S1) 或 None
    """
    block_num = 0
    block_num_each = []
    b = act_seq.shape[0]
    max_kv = max(act_seq)
    # 统计每个 batch 需要的块数
    for cur_s in act_seq:
        cur_block_num = math.ceil(cur_s / block_size)
        block_num_each.append(cur_block_num)
        block_num += cur_block_num
    # block_table shape: (B, max_blocknum_perbatch)
    block_table_shape = [b, math.ceil(max_kv / block_size)]
    # 生成随机排列的物理块号，模拟不连续内存布局
    block_idx_list = torch.arange(0, block_num, 1)
    block_idx_list = block_idx_list[torch.randperm(block_idx_list.size(0))].to(torch.int32)

    # 初始化为 -1（无效块），逐 batch 填入物理块号
    block_table = -torch.ones(block_table_shape, dtype=torch.int32)

    block_table_bidx = 0
    block_idx = 0
    for cur_block in block_num_each:
        for j in range(cur_block):
            block_table[block_table_bidx, j] = block_idx_list[block_idx]
            block_idx += 1
        block_table_bidx += 1

    # 可选: 生成直接索引 KV Cache 的全局 offset
    if need_indices:
        cache_index = -torch.ones((b, s1), dtype=torch.int64)
        for i in range(b):
            cur_act = act_seq[i]
            for j in range(s1):
                pos = cur_act - s1 + j
                block_idx_in_seq = pos // block_size
                global_block_id = block_table[i, block_idx_in_seq]

                offset_in_block = pos % block_size
                global_index = global_block_id * block_size + offset_in_block
                cache_index[i, j] = global_index
    else:
        cache_index = None

    return block_num, block_table, cache_index


def gen_gather_select_attention_golden(dtype, bn1n2s1, is_kn_quant, actual_seq):
    """
    生成 DeepSeek V3/V2 Sparse Flash Attention 的完整测试数据与 golden 输出。

    MLA 结构参数:
        kv_lora_rank = 512   (Key/Value 低秩压缩维度, 即 nope 部分)
        qk_rope_dim  = 64    (旋转位置编码维度, 即 rope 部分)
        d_q = d_k = 576      (kv_lora_rank + qk_rope_dim)
        d_v = 512            (= kv_lora_rank)
        topk = 2048          (每个 token 选出的 KV 数量)
        block_size = 128     (PagedAttention block 大小)

    数据布局:
        q:   (B, S1, N_Q, 576)   [nope(512) | rope(64)]
        kn:  (total_kv, 512)      Key nope 部分, INT8 或 BF16
        kr:  (total_kv, 64)       Key rope 部分, BF16
        kn_scales: (total_kv, 4)  INT8 反量化 scale, FP32 (每 128 元素一组, 512/128=4)

    Args:
        dtype: 数据精度 (torch.bfloat16)
        bn1n2s1: (B, N_Q, N_KV, S1)
        is_kn_quant: 1=INT8 量化, 0=BF16
        actual_seq: 各 batch 实际 KV 序列长度

    Returns:
        input_params: [B, S1, N_Q, N_KV, max_kv_seq, kv_lora_rank, qk_rope_dim,
                       block_num, block_size, topk, is_kn_quant, scalar]
        input_data_map: [q_nope, q_rope, kn, kr, kn_scales, topk_indices,
                         block_table, actual_seq]
        atten_out: golden 注意力输出, shape=(B, S1, N_Q, kv_lora_rank)
    """
    block_size = 128
    torch.manual_seed(42)
    b, n_q, n_kv, s_q = bn1n2s1  # e.g. (4, 128, 1, 2) decode / (1, 128, 1, 256) prefill
    kv_lora_rank = 512            # Key/Value 低秩压缩维度 (nope 部分)
    qk_rope_dim = 64              # 旋转位置编码维度 (rope 部分)
    topk = 2048                   # 每个 token 的 top-k KV 数量
    np.random.seed(None)
    # q head dim = kv_lora_rank + qk_rope_dim = 576
    d_q = kv_lora_rank + qk_rope_dim
    # k head dim = kv_lora_rank + qk_rope_dim = 576
    d_k = kv_lora_rank + qk_rope_dim
    # v head dim = kv_lora_rank = 512
    d_v = kv_lora_rank
    # softmax scale = 1/sqrt(d_q) = 1/sqrt(576)
    scalar = d_q ** -0.5
    if isinstance(actual_seq, int):
        actual_seq = [actual_seq] * b
    elif isinstance(actual_seq, list):
        if len(actual_seq) == b:
            actual_seq = actual_seq
        else:
            raise RuntimeError("unsupported actual_seq list length")
    else:
        raise RuntimeError("unsupported actual_seq data type")

    # ---- 1. 构造输入 shape ----
    shape_q = [b, s_q, n_q, d_q]  # (B, S1, N_Q, 576)

    # 统计所有 batch 的总 block 数
    block_num_per_batch = []
    block_num_min = 0
    block_num = 0
    for actual_seq_tmp in actual_seq:
        block_num_per_batch.append(math.ceil(actual_seq_tmp / block_size))
        block_num_min += math.ceil(actual_seq_tmp / block_size)
    block_num = block_num_min

    # kn: (block_num, block_size, kv_lora_rank) 即 (block_num, 128, 512)
    shape_kn = [block_num, block_size, kv_lora_rank]
    # kr: (block_num, block_size, qk_rope_dim) 即 (block_num, 128, 64)
    shape_kr = [block_num, block_size, qk_rope_dim]

    max_kv_seq = max(actual_seq)
    # 生成 PagedAttention block_table
    block_num, block_table, _ = gen_block_table(torch.tensor(actual_seq), block_size, s_q, need_indices=False)
    # topk_indices: (B, S1, topk) → 后续 reshape 为 (B*S1, N_KV*topk)
    topk_indices = torch.zeros(b, s_q, topk).to(torch.int32)
    slc_actual_seq = []
    for i in range(b):
        slc_actual_seq.append(min(actual_seq[i], topk))

    # 生成 topk_indices: 序列长度 < topk 时顺序取, 否则随机排列取前 topk
    for b_i in range(b):
        for s_q_i in range(s_q):

            if slc_actual_seq[b_i] < topk:
                topk_indices[b_i, s_q_i, :slc_actual_seq[b_i]] = torch.arange(0, slc_actual_seq[b_i])
            else:
                perm = torch.randperm(slc_actual_seq[b_i])
                topk_indices[b_i, s_q_i, :] = perm[:topk]

    topk_indices = topk_indices.reshape(b * s_q, n_kv * topk)

    # ---- 2. 生成随机输入数据 ----
    q_bsnd = gen_uniform_data(shape_q, -1, 1, dtype)            # (B, S1, N_Q, 576) BF16
    kn_bsnd_tmp = gen_uniform_data(shape_kn, -1, 1, dtype)      # (block_num, 128, 512) BF16

    # INT8 量化: 按 128 元素分组求 amax → scale = amax/127 → quant = round(val/scale).clamp(-128,127)
    kn_bsnd_reshape = kn_bsnd_tmp.reshape(block_num * block_size, 4, 128).to(torch.float32)
    kn_scales = kn_bsnd_reshape.abs().amax(dim=-1, keepdim=True).clamp(min=1e-8) / 127.0
    if is_kn_quant == 1:
        kn_quant = kn_bsnd_tmp.reshape(block_num * block_size, 4, 128) / kn_scales
        kn = torch.round(kn_quant).clamp(-128, 127).to(torch.int8)  # (total_kv*4, 128) INT8
    else:
        kn = kn_bsnd_tmp                                                # (block_num, 128, 512) BF16
    kr = gen_uniform_data(shape_kr, -1, 1, dtype)               # (block_num, 128, 64) BF16

    # 展平为 2D: (total_kv, dim)
    kn = kn.reshape(block_num * block_size, kv_lora_rank)       # (total_kv, 512)
    kn_scales = kn_scales.reshape(block_num * block_size, 4)    # (total_kv, 4)
    kr = kr.reshape(block_num * block_size, qk_rope_dim)        # (total_kv, 64)

    # ---- 3. 计算 golden attention（标准 softmax 版本） ----
    params = [block_size, scalar, topk, kv_lora_rank, is_kn_quant]
    input_data = [q_bsnd, kn, kr, kn_scales, topk_indices, block_table, actual_seq]

    s2_tile = 2048
    atten_out, tmp_out = compute_attention_no_flash(input_data, params, s2_tile)

    # ---- 4. 拆分 Query 为 nope + rope, 与 PyPTO kernel 输入格式对齐 ----
    # q_nope: (B*S1*N_Q, 512), q_rope: (B*S1*N_Q, 64)
    q_nope = q_bsnd[:, :, :, :kv_lora_rank]
    q_rope = q_bsnd[:, :, :, kv_lora_rank:]
    q_nope = q_nope.reshape(b * s_q * n_q, kv_lora_rank)
    q_rope = q_rope.reshape(b * s_q * n_q, qk_rope_dim)
    # input params
    input_params = [b, s_q, n_q, n_kv, max_kv_seq, kv_lora_rank, qk_rope_dim, block_num, block_size, topk,
                    is_kn_quant, scalar]
    input_data_map = [q_nope, q_rope, kn, kr, kn_scales, topk_indices, block_table, actual_seq]

    return input_params, input_data_map, atten_out


def do_test_sparse_attention_func(bn1n2s1, actual_seq, input_params, input_data, atten_out, is_p, is_soc_950):
    """
    执行 NPU 端 sparse flash attention 并与 golden 对比。

    根据 (is_p, is_soc_950) 选择不同的 kernel 入口和 TileShape 配置:
        - is_p=True,  is_soc_950=False → sparse_flash_attention_quant_p   (910B Prefill)
        - is_p=False, is_soc_950=False → sparse_flash_attention_quant_d   (910B Decode)
        - is_p=False, is_soc_950=True  → sparse_flash_attention_quant_d_950 (950 Decode)
    """
    b, n1, n2, s1 = bn1n2s1

    device_id = int(os.environ.get('TILE_FWK_DEVICE_ID', 0))
    torch.npu.set_device(device_id)

    # ---- TileShape 配置: 控制各级算子的 tile 切分 ----
    # g_tile: GQA group 维度 tile, 一次处理多少个 query head (group=N_Q/N_KV=128)
    # s_kv_tile: KV 序列维度 tile, 一次处理多少个 KV token
    # gather_vec_tile_shape: [行, 列] Gather 向量算子 tile (从 KV Cache 搬运数据)
    # c1_tile_shape: [M0,M1, K0,K1, N0,N1] C1(Q×K^T) Cube matmul tile
    # v1_tile_shape: [行, 列] V1(Softmax) 向量算子 tile
    # c2_tile_shape: [M0,M1, K0,K1, N0,N1] C2(Softmax×V) Cube matmul tile
    # v2_tile_shape: [行, 列] V2(Flash 归一化更新) 向量算子 tile, 仅 prefill 使用

    if is_p:
        # 910B Prefill 配置 (s1=256)
        tile_config = SaTileShapeConfig(
            g_tile=128,                              # 一次处理 128 个 group
            s_kv_tile=2048,                          # 一次处理 2048 个 KV token
            gather_vec_tile_shape=[32, 512],         # Gather: 32行 × 512列
            c1_tile_shape=[128, 128, 128, 128, 128, 128],  # C1: M=128, K=128, N=128
            v1_tile_shape=[8, 2048],                  # Softmax: 8行 × 2048列
            c2_tile_shape=[128, 128, 128, 128, 128, 128],  # C2: M=128, K=128, N=128 (C1的N轴与C2的K轴一致)
            v2_tile_shape=[64, 128]                   # Flash 归一化更新: 64行 × 128列
        )
    else:
        # 910B Decode 配置 (s1=1 或 2)
        tile_config = SaTileShapeConfig(
            g_tile=128,                              # 一次处理 128 个 group
            s_kv_tile=2048,                          # 一次处理 2048 个 KV token
            gather_vec_tile_shape=[32, 512],         # Gather: 32行 × 512列
            c1_tile_shape=[128, 128, 128, 128, 128, 128],  # C1: M=128, K=128, N=128
            v1_tile_shape=[8, 2048],                  # Softmax: 8行 × 2048列
            c2_tile_shape=[128, 128, 128, 128, 128, 128],  # C2: M=128, K=128, N=128
            v2_tile_shape=[64, 256]                   # Flash 归一化更新: 64行 × 256列
        )
    
    if is_soc_950:
        # 950 Decode 配置 (适配 950 芯片的 UB/L1 容量)
        tile_config = SaTileShapeConfig(
            g_tile=128,                              # 一次处理 128 个 group
            s_kv_tile=2048,                          # 一次处理 2048 个 KV token
            gather_vec_tile_shape=[64, 512],         # Gather: 64行 × 512列 (950 UB 更大, 行翻倍)
            c1_tile_shape=[128, 128, 128, 128, 64, 64],    # C1: M=128, K=128, N=64 (950 N 轴减半)
            v1_tile_shape=[4, 2048],                  # Softmax: 4行 × 2048列 (行减半)
            c2_tile_shape=[128, 128, 128, 128, 128, 128],  # C2: M=128, K=128, N=128
            v2_tile_shape=[64, 256]                   # Flash 归一化更新: 64行 × 256列
        )

    b, s1, n_q, n_kv, max_kv_seq, kv_lora_rank, qk_rope_dim, block_num, block_size, topk, \
        is_kn_quant, softmax_scale = input_params
    q_nope, q_rope, kn, kr, kn_scales, topk_indices, block_table, kv_actual_seqs = input_data
    kv_act_seqs = torch.tensor(actual_seq, dtype=torch.int32)

    # ---- 将所有输入 tensor 搬运到 NPU ----
    q_nope_npu = q_nope.npu()
    q_rope_npu = q_rope.npu()
    kn_npu = kn.npu()
    kr_npu = kr.npu()
    kn_scales_npu = kn_scales.npu()
    topk_indices_npu = topk_indices.npu()
    block_table_npu = block_table.npu()
    kv_act_seqs_npu = kv_act_seqs.npu()
    pto_inputs = [q_nope_npu, q_rope_npu, kn_npu, kr_npu, kn_scales_npu, topk_indices_npu, block_table_npu,
                  kv_act_seqs_npu]

    # 输出 tensor, shape=(B, S1, N_Q, kv_lora_rank), BF16
    calc_attention_out = torch.zeros([b, s1, n_q, kv_lora_rank], dtype=torch.bfloat16)
    calc_attention_out_npu = calc_attention_out.npu()
    pto_outputs = [calc_attention_out_npu]

    # max_blocknum_perbatch: 用于 block_table 的列数
    max_blocknum_perbatch = math.ceil(max_kv_seq / block_size)

    # ---- 根据 (is_p, is_soc_950) 选择对应的 JIT kernel ----
    if is_p and not is_soc_950:
        # 910B Prefill: Flash Attention (online softmax)
        sparse_flash_attention_quant_p(*pto_inputs, *pto_outputs, n_q, n_kv, softmax_scale, topk, block_size, \
            max_blocknum_perbatch, tile_config)
    elif not is_p and not is_soc_950:
        # 910B Decode: 标准 softmax
        sparse_flash_attention_quant_d(*pto_inputs, *pto_outputs, n_q, n_kv, softmax_scale, topk, block_size, \
            max_blocknum_perbatch, tile_config)
    else:
        # 950 Decode: 标准 softmax (适配 950 芯片)
        sparse_flash_attention_quant_d_950(*pto_inputs, *pto_outputs, n_q, n_kv, softmax_scale, topk, block_size, \
            max_blocknum_perbatch, tile_config)
    # 等待 NPU 计算完成
    torch_npu.npu.synchronize()
    # 精度对比: atol=0.0001, rtol=0.005, 最多报告 100 个误差点
    compare(calc_attention_out_npu.cpu(), atten_out, "atten_out", atol=0.0001, rtol=0.005, max_error_count=100)


def get_case_config(case_name: str):
    """
    测试用例参数配置。
    每个用例: ((B, N_Q, N_KV, S1), is_kn_quant, actual_seq, is_soc_950)

    命名规则: sfa_{dtype}_b{B}_s{S1}_seq{KV_LEN}_{quant}_{mode}
      - mode 后缀: d=decode, p=prefill
      - quant 后缀: int8=INT8 量化 Key, bf16=纯 BF16 Key
      - seq 后缀: total=变长序列, per=等长序列, 64K=65536
    """
    test_case_config = {
        # 变长序列 decode (INT8): B=4, S1=2, 各 batch 序列长度不同
        "sfa_bf16_b4_s2_seq64K_total_int8_d": (
            (4, 128, 1, 2), 1, [65536, 16381, 666, 15], 0
        ),
        # 等长序列 decode (INT8): B=4, S1=2, 所有 batch seq=65536
        "sfa_bf16_b4_s2_seq64K_per_int8_d": (
            (4, 128, 1, 2), 1, [65536] * 4, 0
        ),
        # 等长序列 decode (BF16): B=4, S1=2, Key 无量化
        "sfa_bf16_b4_s2_seq64K_per_bf16_d": (
            (4, 128, 1, 2), 0, [65536] * 4, 0
        ),
        # prefill (INT8): B=1, S1=256, 单 batch 长序列
        "sfa_bf16_b1_s256_seq64K_int8_p": (
            (1, 128, 1, 256), 1, [65536], 0
        ),
        # 950 decode (BF16): B=4, S1=2, 950 芯片适配
        "sfa_bf16_b4_s2_seq64K_per_int8_d_950": (
            (4, 128, 1, 2), 0, [65536] * 4, 1
        ),
    }
    case_config = test_case_config.get(case_name)
    return case_config


def do_test_sfa_entry(case_name: str, is_p: bool, is_soc_950: bool):
    """
    测试入口: 根据用例名获取参数 → 生成 golden → 执行 NPU 计算 → 精度对比。
    """
    case_config = get_case_config(case_name)
    if not case_config:
        logging.error("Can't get func to gen golden, Case(%s)", case_name)
        return False
    bn1n2s1, is_kn_quant, actual_seq, is_soc_950 = case_config

    # 生成 golden 数据与参考输出
    input_params, input_data, atten_out = gen_gather_select_attention_golden(
        torch.bfloat16, bn1n2s1, is_kn_quant, actual_seq
    )
    # 执行 NPU kernel 并与 golden 对比
    do_test_sparse_attention_func(
        bn1n2s1, actual_seq, input_params, input_data, atten_out, is_p, is_soc_950
    )
    return True


@pytest.mark.soc("950", "910")
def test_sfa_bf16_b4_s2_seq64k_total_int8_d():
    """
    Decode 变长序列测试 (INT8 量化 Key)
    配置: B=4, S1=2, N_Q=128, N_KV=1, topk=2048
    序列长度: [65536, 16381, 666, 15] — 各 batch 差异大，验证变长处理逻辑
    芯片: 910B / 950
    模式: is_p=False (decode), is_soc_950=False
    """
    do_test_sfa_entry("sfa_bf16_b4_s2_seq64K_total_int8_d", is_p=False, is_soc_950=False)


@pytest.mark.skip(reason="perf")
def test_sfa_bf16_b4_s2_seq64k_per_int8_d():
    """
    Decode 等长序列测试 (INT8 量化 Key)
    配置: B=4, S1=2, N_Q=128, N_KV=1, topk=2048
    序列长度: [65536] × 4 — 所有 batch 等长
    芯片: 910B / 950
    模式: is_p=False (decode), is_soc_950=False
    """
    do_test_sfa_entry("sfa_bf16_b4_s2_seq64K_per_int8_d", is_p=False, is_soc_950=False)


@pytest.mark.soc("950")
@pytest.mark.skip(reason="perf")
def test_sfa_bf16_b4_s2_seq64k_per_int8_d_950():
    """
    Decode 等长序列测试 — 950 芯片适配 (BF16 Key)
    配置: B=4, S1=2, N_Q=128, N_KV=1, topk=2048
    序列长度: [65536] × 4
    芯片: 950 (专用 TileShape: gather 行翻倍, C1 N 轴减半, V1 行减半)
    模式: is_p=False (decode), is_soc_950=True
    """
    do_test_sfa_entry("sfa_bf16_b4_s2_seq64K_per_int8_d_950", is_p=False, is_soc_950=True)


@pytest.mark.skip(reason="bf16 perf")
def test_sfa_bf16_b4_s2_seq64k_per_bf16_d():
    """
    Decode 等长序列测试 (纯 BF16 Key, 无量化)
    配置: B=4, S1=2, N_Q=128, N_KV=1, topk=2048
    序列长度: [65536] × 4
    芯片: 910B / 950
    模式: is_p=False (decode), is_soc_950=False
    目的: 验证 Key 未量化时 kernel 的 BF16 路径（Gather → 直接使用，不经反量化）
    """
    do_test_sfa_entry("sfa_bf16_b4_s2_seq64K_per_bf16_d", is_p=False, is_soc_950=False)


@pytest.mark.skip(reason="large test case")
def test_sfa_bf16_b1_s256_seq64k_int8_p():
    """
    Prefill 测试 (INT8 量化 Key, Flash online softmax)
    配置: B=1, S1=256, N_Q=128, N_KV=1, topk=2048
    序列长度: [65536]
    芯片: 910B
    模式: is_p=True (prefill), is_soc_950=False
    特点: S1=256 较大，使用 Flash Attention 增量更新 oi/li/mi 跨 tile 归一化
    """
    do_test_sfa_entry("sfa_bf16_b1_s256_seq64K_int8_p", is_p=True, is_soc_950=False)


if __name__ == "__main__":
    logging.basicConfig(
        format='%(asctime)s - %(filename)s:%(lineno)d - %(levelname)s: %(message)s',
        level=logging.INFO
    )
    # ---- 直接运行入口: 取消注释即可执行对应用例 ----
    # 当前激活: BF16 等长 decode (无量化, 数据量最小, 适合快速验证)
    # test_sfa_bf16_b4_s2_seq64k_total_int8_d()   # 变长 INT8 decode (默认 pytest 用例)
    # test_sfa_bf16_b4_s2_seq64k_per_int8_d()     # 等长 INT8 decode
    # test_sfa_bf16_b1_s256_seq64k_int8_p()       # prefill INT8 (S1=256, 耗时较长)
    # test_sfa_bf16_b4_s2_seq64k_per_int8_d_950() # 950 decode BF16
    test_sfa_bf16_b4_s2_seq64k_per_bf16_d()         # 等长 BF16 decode (无量化)

Start time: 2026-05-21 02:45:56
Total Core:72
Total Task Count:576
|--Fake Task Count:32
Parse Swim json and Topo json Data End

-------- AICORE Prof Summary --------
AICore End-to-End Time: 115.64
AICore Utilization:     47.70%
-------------------------------------
Generate Bubble Analysis Report: output/output_20260521_104536_947389_578358_C0A800C5/bubble_analysis.log
Convert To perfetto trace End
Open the trace at https://ui.perfetto.dev/ 
Output:  output/output_20260521_104536_947389_578358_C0A800C5/merged_swimlane.json
End time: 2026-05-21 02:45:57
Time taken: 0 secs
compare success !!!!
